# 9: OPTICS, Dimensionality Reduction, and PCA

**Author:** Dr Rob Lyon 

**Version:** 1.0

## Code & License
Copyright © 2026 Robert Lyon. All Rights Reserved.

This notebook and all associated materials are the intellectual property of the author.

Permission is granted solely to read, study, and analyse this material for personal educational purposes. No other rights are granted.

Without the prior written consent of the author, you may not:

* Copy, reproduce, redistribute, publish, transmit, or display this work in whole or in part.
* Modify, adapt, transform, translate, or create derivative works based on this material.
* Incorporate any portion of this work into another project, publication, product, model, dataset, or codebase.
* Use this material for commercial purposes.
* Remove or alter this copyright notice.

All intellectual property rights, including copyright and any derivative rights, remain exclusively vested in the author.

Access to this material does not constitute a transfer of ownership, license, or any other intellectual property rights except as expressly stated above.

---

**Note:**

This notebook is designed to be viewed from within JupyterLab. The Table of Contents and "Back to Table of Contents" links rely on JupyterLab's anchor handling to jump between sections.

If you open this notebook in another tool, such as PyCharm or Visual Studio Code, these anchor links may not work as expected. You can still navigate the notebook in those tools by using the headings directly, most editors provide an outline or contents panel that lists them for you.

---

## Table of Contents
1. [Introduction](#1.-Introduction)
2. [Useful Links](#2.-Useful-Links)
3. [Libraries and Environment Setup](#3.-Libraries-and-Environment-Setup)
4. [OPTICS: Density-Based Clustering with Varying Density](#4.-OPTICS:-Density-Based-Clustering-with-Varying-Density)
   - [4.1 Core Concept: Reachability Distance](#4.1-Core-Concept:-Reachability-Distance)
   - [4.2 OPTICS with scikit-learn](#4.2-OPTICS-with-scikit-learn)
5. [Moving Beyond Clustering: Dimensionality Reduction](#5.-Moving-Beyond-Clustering:-Dimensionality-Reduction)
6. [The Building Blocks of PCA](#6.-The-Building-Blocks-of-PCA)
   - [6.1 Variance: How Spread Out Is a Variable?](#6.1-Variance:-How-Spread-Out-Is-a-Variable?)
   - [6.2 Covariance: Do Two Variables Move Together?](#6.2-Covariance:-Do-Two-Variables-Move-Together?)
   - [6.3 The Covariance Matrix: All Pairwise Relationships at Once](#6.3-The-Covariance-Matrix:-All-Pairwise-Relationships-at-Once)
7. [Vectors and Matrices](#7.-Vectors-and-Matrices)
   - [7.1 Vector and Matrix Multiplication](#7.1-Vector-and-Matrix-Multiplication)
8. [Eigenvectors and Eigenvalues](#8.-Eigenvectors-and-Eigenvalues)
   - [8.1 Where Do Eigenvalues Come From? A Walkthrough](#8.1-Where-Do-Eigenvalues-Come-From?-A-Walkthrough)
9. [Principal Component Analysis (PCA)](#9.-Principal-Component-Analysis-(PCA))
   - [9.1 The Full Algorithm](#9.1-The-Full-Algorithm)
10. [PCA with scikit-learn](#10.-PCA-with-scikit-learn)
    - [10.1 Explained Variance and the Scree Plot](#10.1-Explained-Variance-and-the-Scree-Plot)
    - [10.2 Loadings: What Do the PCs Mean?](#10.2-Loadings:-What-Do-the-PCs-Mean?)
11. [PCA with Breast Cancer Dataset](#11.-PCA-with-Breast-Cancer-Dataset)
    - [11.1 Scree Plot and Variance Explained](#11.1-Scree-Plot-and-Variance-Explained)
    - [11.2 The Biplot: Data in PC Space](#11.2-The-Biplot:-Data-in-PC-Space)
    - [11.3 Classifying in PCA Space — The Right Approach](#11.3-Classifying-in-PCA-Space-—-The-Right-Approach)
    - [11.4 Common PCA Mistakes](#11.4-Common-PCA-Mistakes)
12. [Summary](#12.-Summary)
13. [References](#13.-References)

---

## 1. Introduction

Welcome to **Notebook 9**. It completes the unsupervised learning arc and introduces a second major branch of the field: **dimensionality reduction**.

### Aim

This notebook extends your understanding of density-based clustering by introducing OPTICS, a generalisation of DBSCAN that copes with clusters of varying density, and then builds a thorough, ground-up understanding of **Principal Component Analysis (PCA)**. Rather than jumping straight to the algorithm, we develop each mathematical ingredient in turn: variance, covariance, the covariance matrix, vectors, eigenvectors, and eigenvalues. Once all the pieces are in place, PCA emerges as the natural consequence of combining them.

### Learning Objectives

By the end of this notebook you should be able to:

- Explain how OPTICS extends DBSCAN by using a continuous reachability distance instead of a fixed radius.
- Read and interpret an OPTICS reachability plot to identify clusters and noise.
- Define variance and covariance, and implement both from scratch in Python.
- Construct a covariance matrix manually and verify it against NumPy.
- Explain what an eigenvector and eigenvalue are in geometric terms.
- Derive eigenvalues from the characteristic polynomial for a small matrix.
- Implement the full PCA algorithm from scratch (standardise → covariance matrix → eigenvectors → project).
- Use a scree plot and cumulative variance curve to choose the number of principal components.
- Interpret PCA loadings to understand what each component represents.
- Correctly apply PCA in a train/test pipeline, avoiding the data-leakage pitfall.

[Back to Table of Contents](#Table-of-Contents)

---


## 2. Useful Links

| Link | Description |
|------|-------------|
| [scikit-learn: OPTICS](https://scikit-learn.org/stable/modules/clustering.html#optics) | Official documentation for the `OPTICS` class, including parameter descriptions and worked examples. Useful alongside Section 4. |
| [scikit-learn: PCA](https://scikit-learn.org/stable/modules/decomposition.html#pca) | Documentation for `sklearn.decomposition.PCA`, covering parameters, attributes such as `explained_variance_ratio_`, and usage examples. Useful alongside Sections 9–10. |
| [NumPy: `numpy.linalg.eig`](https://numpy.org/doc/stable/reference/generated/numpy.linalg.eig.html) | Reference for computing eigenvalues and eigenvectors numerically. Also see `numpy.linalg.eigh` for symmetric matrices. Used in Sections 8 and 9. |
| [NumPy: `numpy.cov`](https://numpy.org/doc/stable/reference/generated/numpy.cov.html) | Reference for the NumPy covariance function, explaining the `ddof` parameter (Bessel's correction). Used in Section 6. |
| [Seaborn: `heatmap`](https://seaborn.pydata.org/generated/seaborn.heatmap.html) | Reference for the heatmap function used to visualise covariance matrices in Section 6.3. |


[Back to Table of Contents](#Table-of-Contents)



---

## 3. Libraries and Environment Setup

### 3.1 Working Environment

To run the code in this notebook, you will need a Python environment with the required libraries listed below installed.

The easiest way to get started is to use the **Project IO** Docker container, which provides a fully configured and tested environment containing all required Python packages and supporting dependencies. Instructions for downloading and running the container can be found in the **project-io** GitHub repository:

https://github.com/scienceguyrob/project-io

Using the Docker container ensures that the notebooks run in exactly the same environment in which they were developed and tested.

If you prefer not to use Docker, a good alternative is to install the [Anaconda Distribution](https://www.anaconda.com/products/distribution). You can then create a new Python environment and install the required libraries using either `conda install` or `pip install`.

### 3.2 Libraries Used in This Notebook

All sections of this notebook require external libraries. The table below summarises each one.

| Library | Purpose | Documentation |
|---------|---------|---------------|
| **NumPy** (`numpy`) | Fast numerical arrays, linear algebra (`np.cov`, `np.linalg.eig`, `np.linalg.eigh`), and random number generation. | [numpy.org](https://numpy.org/doc/stable/) |
| **Pandas** (`pandas`) | Creating and displaying tabular summaries of loadings and results. | [pandas.pydata.org](https://pandas.pydata.org/docs/) |
| **Matplotlib** (`matplotlib.pyplot`) | Core plotting library used for scatter plots, bar charts, line plots, and annotation arrows throughout. | [matplotlib.org](https://matplotlib.org/stable/) |
| **Seaborn** (`seaborn`) | Higher-level statistical visualisation; used for heatmaps of covariance matrices. | [seaborn.pydata.org](https://seaborn.pydata.org/) |
| **scikit-learn** (`sklearn`) | Provides `OPTICS`, `DBSCAN`, `PCA`, `StandardScaler`, `LogisticRegression`, and built-in datasets (Iris, Breast Cancer, Wine). | [scikit-learn.org](https://scikit-learn.org/stable/) |
| **ipywidgets** (`ipywidgets`) | Adds interactive UI controls (sliders, dropdowns, etc.) to Jupyter notebooks. We use it to create sliders for adjusting plot parameters in real time. | [ipywidgets.readthedocs.io](https://ipywidgets.readthedocs.io/en/stable/) |
| **ipympl** (`matplotlib widget` backend) | Enables interactive, live-updating Matplotlib figures inside Jupyter. Required for smooth slider updates without redrawing the entire plot. | [matplotlib.org/ipympl](https://matplotlib.org/ipympl/) |

### 3.3 Data

This notebook uses two datasets.

#### **Breast Cancer Wisconsin (Diagnostic) Dataset**

We used this dataset before in notebook 7. The dataset is built directly into scikit-learn and requires no external file download.

The dataset contains 569 observations, each representing a tumour biopsy. For every biopsy, 30 numeric features are computed from a digitised image of a needle biopsy sample taken from a breast mass. The 30 features describe characteristics of the cell nuclei present in the image. The features are organised into three groups of 10, each group computed differently from the same underlying measurements:

| Feature group | Example features | How computed |
|---|---|---|
| `mean` | `mean radius`, `mean texture` | Mean value across all nuclei in the image |
| `se` (standard error) | `radius error`, `texture error` | Standard error of the measurement |
| `worst` | `worst radius`, `worst texture` | Mean of the three largest values |

The target variable is binary: **0 = Malignant**, **1 = Benign**. The dataset is class-imbalanced, with approximately 63% Benign and 37% Malignant cases. The full list of features in the dataset is provided below for clarity.

| Variable | Type | Range / Values | Description |
|---|---|---|---|
| `mean radius` | Float | ~6–28 | Average distance from the centre of a nucleus to its boundary; larger values indicate larger nuclei |
| `mean texture` | Float | ~9–40 | Variation in grey-scale intensity within the nucleus image; higher values indicate a less uniform texture |
| `mean perimeter` | Float | ~44–190 | Average length of the nucleus boundary |
| `mean area` | Float | ~140–2500 | Average area enclosed by the nucleus boundary |
| `mean smoothness` | Float | ~0.05–0.16 | Average variation in radius lengths; higher values indicate less smooth, more irregular boundaries |
| `mean compactness` | Float | ~0.02–0.35 | Measure of how tightly packed the nucleus shape is, calculated from perimeter and area; larger values indicate more complex shapes |
| `mean concavity` | Float | ~0–0.43 | Average severity of inward-curving indentations along the nucleus boundary; higher values indicate deeper concave regions |
| `mean concave points` | Float | ~0–0.20 | Average number and prominence of points where the boundary curves inward |
| `mean symmetry` | Float | ~0.10–0.30 | Degree to which the nucleus shape is symmetrical around its centre |
| `mean fractal dimension` | Float | ~0.05–0.10 | Measure of boundary complexity; higher values indicate a more jagged or irregular contour |
| `radius error` | Float | ~0.1–2.9 | Standard error (estimated variability) of radius measurements across nuclei |
| `texture error` | Float | ~0.4–5.0 | Standard error of texture measurements |
| `perimeter error` | Float | ~0.8–22 | Standard error of perimeter measurements |
| `area error` | Float | ~6–542 | Standard error of area measurements |
| `smoothness error` | Float | ~0.002–0.03 | Standard error of smoothness measurements |
| `compactness error` | Float | ~0.002–0.14 | Standard error of compactness measurements |
| `concavity error` | Float | ~0–0.40 | Standard error of concavity measurements |
| `concave points error` | Float | ~0–0.05 | Standard error of concave point measurements |
| `symmetry error` | Float | ~0.008–0.08 | Standard error of symmetry measurements |
| `fractal dimension error` | Float | ~0.001–0.03 | Standard error of fractal dimension measurements |
| `worst radius` | Float | ~7–36 | Mean of the three largest radius values observed, representing the most extreme nucleus sizes |
| `worst texture` | Float | ~12–50 | Mean of the three largest texture values observed |
| `worst perimeter` | Float | ~50–250 | Mean of the three largest perimeter values observed |
| `worst area` | Float | ~185–4250 | Mean of the three largest area values observed |
| `worst smoothness` | Float | ~0.07–0.22 | Mean of the three largest smoothness values observed |
| `worst compactness` | Float | ~0.03–1.10 | Mean of the three largest compactness values observed |
| `worst concavity` | Float | ~0–1.30 | Mean of the three largest concavity values, capturing the most pronounced inward boundary indentations |
| `worst concave points` | Float | ~0–0.30 | Mean of the three largest concave point measurements observed |
| `worst symmetry` | Float | ~0.15–0.70 | Mean of the three largest symmetry measurements observed |
| `worst fractal dimension` | Float | ~0.06–0.21 | Mean of the three largest fractal dimension measurements observed |
| `target` | Integer | 0, 1 | Diagnosis label: 0 = Malignant (cancerous), 1 = Benign (non-cancerous) |

#### **Iris Dataset**

This is also built directly into scikit-learn and requires no external file download.

The dataset contains 150 observations, each representing a single iris flower. For every flower, 4 numeric features are recorded, all measured in centimetres using physical rulers on real flower specimens. The dataset is perfectly balanced across three species, with exactly 50 observations each.

| Variable | Type | Range / Values | Description |
|---|---|---|---|
| `sepal length (cm)` | Float | ~4.3–7.9 | Length of the sepal — the leaf-like structure beneath the petals that protects the flower bud |
| `sepal width (cm)` | Float | ~2.0–4.4 | Width of the sepal |
| `petal length (cm)` | Float | ~1.0–6.9 | Length of the petal |
| `petal width (cm)` | Float | ~0.1–2.5 | Width of the petal |
| `target` | Integer | 0, 1, 2 | Species label: 0 = setosa, 1 = versicolor, 2 = virginica |

Unlike the Breast Cancer dataset, where the `mean`/`se`/`worst` groupings reflect repeated measurements summarised in different ways, every feature in the Iris dataset is a single direct physical measurement of one flower. There's no aggregation or repeated sampling involved. This makes Iris a useful first dataset for PCA precisely because it's small and easy to interpret: with only 4 features, it's still possible to look at the data directly and sanity-check what each principal component is doing, before moving on to higher-dimensional datasets like Breast Cancer where this becomes impractical.

### 3.4 Imports

All imports for this notebook are placed in the single cell below. Keeping every import together at the top is a deliberate best practice: it makes dependencies immediately visible and prevents errors that arise when an import cell gets skipped.

> ⚠️ **You must run the cell below before executing any other code in this notebook.** If you skip it, all subsequent code cells will raise a `NameError`.

In [ ]:
# Listing 1.
# ─────────────────────────────────────────────────────────────────────────────
# All library imports for this notebook are placed here for clarity.
# Run this cell first before executing any code.
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np                           # numerical arrays and linear algebra
import pandas as pd                           # tabular data and display
import matplotlib.pyplot as plt               # plotting
import matplotlib.patches as mpatches        # custom legend elements


import matplotlib
matplotlib.rcParams['figure.max_open_warning'] = 50


import seaborn as sns                         # statistical visualisation (heatmaps)
from sklearn.cluster import DBSCAN, OPTICS   # density-based clustering algorithms
from sklearn.datasets import (
    make_moons,           # synthetic half-moon dataset
    load_breast_cancer,   # 30-feature cancer dataset
    load_iris,            # 4-feature flower dataset
    load_wine,            # 13-feature wine dataset
)
from sklearn.preprocessing import StandardScaler    # zero-mean unit-variance scaling
from sklearn.decomposition import PCA               # Principal Component Analysis
from sklearn.linear_model import LogisticRegression # classifier used in pipeline demos
from sklearn.model_selection import train_test_split # train/test splitting
from sklearn.metrics import accuracy_score           # classification evaluation

# Global plot settings
rng = np.random.default_rng(42)            # reproducible random number generator

# Confirm successful import
print('Libraries loaded successfully.')


---


## 4. OPTICS: Density-Based Clustering with Varying Density

🔙 [Back to Table of Contents](#table-of-contents)

If you recall from notebook 8, DBSCAN classifies every point as either a core point, a border point, or noise based on two fixed parameters: $\varepsilon$ (neighbourhood radius) and `minPts` (minimum neighbour count). This lets it discover clusters of arbitrary shape and identify outliers without needing a parameter like $k$ (the number of clusters) specified in advance. But DBSCAN has a fundamental limitation: $\varepsilon$ is fixed across the entire dataset. If one cluster is tight and dense while another is loose and sparse, no single $\varepsilon$ works well for both. A value that correctly separates the dense cluster from noise will either swallow the sparse cluster into noise too, or merge it with neighbouring points that shouldn't belong to it.

Figure 1 below makes this limitation concrete. The dataset contains two clusters built to have very different densities: a **dense cluster** centred on the left, where points are packed tightly together, and a **sparse cluster** centred on the right, where the same number of points is spread over a much larger area, along with a handful of scattered points that are genuine outliers (true noise). A single slider controls $\varepsilon$; `minPts` is fixed at 5 throughout, so you can isolate the effect of $\varepsilon$ on its own.

As you move the slider, DBSCAN is refit on the data with the new $\varepsilon$, and every point is recoloured according to the cluster label it receives: blue for points assigned to the dense cluster, red for points assigned to the sparse cluster, purple if the two clusters have merged into one, and grey crosses for points labelled as noise ($-1$). The annotation panel on the right reports, in plain terms, what happened to each cluster (correctly recovered, fragmenting into noise, lost to noise entirely, or merged with the other cluster), along with the fraction of each cluster's points currently labelled as noise.

Start at the default $\varepsilon = 0.45$ and work through the following experiments:

* **Decreasing $\varepsilon$:** the dense cluster remains intact, since its points are close enough together to satisfy `minPts` even at a small radius. The sparse cluster, however, begins to fall apart: its points are too far from one another to meet the neighbour count, so they are reclassified as noise one by one. Push $\varepsilon$ low enough and the entire sparse cluster disappears into noise while the dense cluster still looks perfect.

* **Increasing $\varepsilon$:** the sparse cluster now has enough room for its points to find `minPts` neighbours, and it starts to be recovered properly. But keep increasing $\varepsilon$ and watch the dense cluster's neighbourhood circles grow large enough to reach across the gap into the sparse cluster's territory. At this point the annotation panel will report a merge, and both clusters collapse into a single purple blob.

The key thing to notice is that there is no value of $\varepsilon$ in between that makes the annotation panel report "Correctly recovered" for *both* clusters with a low noise fraction at the same time as keeping them clearly separate. A small window may exist where both look reasonable, but it is narrow, and either side of it one cluster is sacrificed for the other. This is the structural problem OPTICS is designed to solve: rather than picking a single $\varepsilon$ and living with this trade-off, it considers the full range of $\varepsilon$ values at once.


In [ ]:
# Listing 2.
%matplotlib widget
from visualisations.Figure_1 import show
show()

---

**OPTICS** (Ordering Points To Identify the Clustering Structure) addresses this by extending the same core ideas, core points, neighbourhoods, density, but replacing DBSCAN's binary core/non-core decision with a continuous numerical measure of how reachable each point is from its neighbours. Rather than committing to one $\varepsilon$ value upfront, OPTICS processes every point in an order determined by this **reachability distance**, which adapts automatically to the local density around each point.

The result of this ordering is a **reachability plot**, a single chart that effectively encodes the clustering structure across *every* possible value of $\varepsilon$ simultaneously. On this plot:

- Low valleys correspond to dense clusters: points inside a cluster are easily reachable from their neighbours, so their reachability distance is small.
- High peaks correspond to noise or sparse regions between clusters: points here are far from any dense neighbourhood, so their reachability distance is large.
- The sharp rises and falls between valleys mark cluster boundaries.

Once you have this plot, you can extract clusters at *any* density threshold by drawing a horizontal cutoff line across it: points whose reachability distance falls below the line belong to a cluster, points above it are treated as noise. Because the plot already encodes the full range of densities, this is equivalent to running DBSCAN at every possible $\varepsilon$ at once and simply choosing which result to read off afterwards.

---

Figure 2 below fits OPTICS once on the same dataset from Figure 1 (one dense cluster, one sparse cluster, and a handful of scattered noise points) and shows the result as two linked panels. The bottom panel is the **reachability plot**: each bar represents one point, placed in the order OPTICS visits it, with a height equal to that point's reachability distance. The top panel shows the same points in feature space.

A dashed horizontal line on the reachability plot is a **cutoff threshold** that you can drag up and down with the mouse. Points whose bar falls *below* the line belong to whichever valley they sit in; points *above* the line are treated as noise. The top scatter panel is recoloured to match (blue for points in the dense-cluster valley, red for points in the sparse-cluster valley, and grey for anything above the cutoff) so you can see directly how a horizontal cut through the reachability plot corresponds to a clustering of the actual data.

Start with the cutoff roughly a third of the way up the plot and try the following:

* **Lowering the cutoff:** only the lowest, narrowest valley, the dense cluster, stays below the line. Everything else, including the sparse cluster, is pushed above the cutoff and turns grey (noise). This mirrors what happened in Figure 1 at small $\varepsilon$: the dense cluster survives, the sparse cluster does not.

* **Raising the cutoff slightly:** the sparse cluster's valley, which sits higher than the dense cluster's because its points are further apart, now also dips below the line. Both clusters appear in their respective colours at the same time, something Figure 1 was never able to achieve with a single $\varepsilon$.

* **Raising the cutoff further:** eventually the cutoff rises above the peak *between* the two valleys, the sparse inter-cluster region. At that point the two valleys are no longer separated by anything above the line, so OPTICS treats the whole stretch as one long run, and the colouring on both panels shows the clusters merging, the same outcome as a too-large $\varepsilon$ in Figure 1.

The key idea is that you didn't have to refit anything to move between these three outcomes: OPTICS was fitted only once. Every possible DBSCAN result, for every possible $\varepsilon$, is already encoded somewhere in this single reachability plot; choosing a cutoff is just choosing which of those results to read off.

In [ ]:
# Listing 3.
%matplotlib widget
from visualisations.Figure_2 import show
show()

### 4.1 Core Concept: Reachability Distance

To build the reachability plot, OPTICS reuses a concept from DBSCAN, the **core distance**, and combines it with the straightforward Euclidean (or other chosen metric) distance between two points, $d(p, o)$, to define the reachability distance.

The **core distance** of a point $o$, written $\text{core-dist}_{\text{minPts}}(o)$, is the distance from $o$ to its `minPts`-th nearest neighbour. It answers the question "how far do I have to look around $o$ before I've found `minPts` neighbours?" A small core distance means $o$ sits in a dense region; a large core distance means $o$ sits in a sparse one.

The **reachability distance** of a point $p$ with respect to another point $o$ is then defined as:

$$
\text{reach-dist}_{\text{minPts}}(p, o) = \max\bigl(\text{core-dist}_{\text{minPts}}(o),\ d(p, o)\bigr)
$$

In words: the reachability distance from $o$ to $p$ is whichever is larger, the actual distance between them, or $o$'s core distance.

> 💡 **Intuition:** Why take the maximum rather than just using the actual distance $d(p, o)$? Imagine $o$ sits in a tightly packed cluster, and $p$ is one of its neighbours: say $d(p, o) = 0.01$, while $o$'s core distance (the distance to its `minPts`-th nearest neighbour) is $0.05$. Reporting the true distance of $0.01$ would suggest $p$ is *exceptionally* easy to reach from $o$, easier than most of the other points packed just as tightly around $o$, whose distances to $o$ are closer to $0.05$. But this difference is essentially noise: all of these points are equally "inside" $o$'s dense neighbourhood, and the small variations in their exact distances don't reflect any real difference in density. Flooring the reachability distance at $\text{core-dist}(o) = 0.05$ removes this noise: every point inside $o$'s neighbourhood is assigned the same reachability value, $0.05$, regardless of whether it happens to sit slightly closer or further away. It's this flattening that produces the smooth, flat-bottomed valleys in the reachability plot, rather than a jagged line where each point's value depends on tiny, meaningless differences in distance.

---

Figure 3 lets you explore the reachability distance formula directly, point by point. The plot shows a fixed neighbourhood of six grey points surrounding an anchor point $o$ (blue). A dashed gold circle marks $o$'s **core distance**, the distance to its `minPts`-th nearest neighbour among those six fixed points, here with `minPts` $= 4$. Because this circle depends only on the fixed points, it never moves, no matter what else happens on the plot.

One additional point, $p$ (red), is draggable: click and drag it anywhere on the plot. As you move $p$, a dotted line tracks the actual distance $d(p, o)$, and a solid purple circle around $o$ shows the resulting reachability distance, $\text{reach-dist}(p, o) = \max(\text{core-dist}(o), d(p, o))$. The annotation panel on the right updates live with the exact numerical values of $\text{core-dist}(o)$, $d(p, o)$, and $\text{reach-dist}(p, o)$, and tells you which of the two terms is currently determining the result.

Try the following:

* **Drag $p$ inside the gold circle:** notice that the purple reachability circle stays locked to the gold core-distance circle, even as $d(p, o)$ (the dotted line) changes length. The annotation panel confirms that $\text{reach-dist}(p, o) = \text{core-dist}(o)$ regardless of exactly where $p$ sits, as long as it's inside, its precise position doesn't matter.

* **Drag $p$ outside the gold circle:** now the purple circle grows and shrinks together with the dotted line: $d(p, o)$ has become larger than $\text{core-dist}(o)$, so it's the one that determines $\text{reach-dist}(p, o)$.

* **Drag $p$ slowly across the gold circle's boundary:** watch the moment the purple circle "detaches" from the gold one and starts following $p$ instead. This is the exact transition described in the intuition box above: points packed inside $o$'s dense neighbourhood all get the same reachability value, while points further out get penalised in proportion to how far away they actually are.

In [ ]:
# Listing 4.
%matplotlib widget
from visualisations.Figure_3 import show
show()

---

### 4.2 OPTICS with scikit-learn

The reachability plot in Figure 2 was produced by scikit-learn's `OPTICS` class, which computes the ordering and reachability distances for you. The interface deliberately mirrors `DBSCAN` from Notebook 8: both are density-based and both accept `min_samples`, but `OPTICS` does not require `eps` to be fixed in advance, and it exposes the full reachability structure rather than committing immediately to a single set of cluster labels.

---

#### Creating and fitting the model

Here's how to fit the model:

```python
from sklearn.cluster import OPTICS
from sklearn.preprocessing import StandardScaler

# Standardise features before fitting — OPTICS, like DBSCAN, computes
# neighbourhoods directly from distances, so unscaled features will
# distort those distances exactly as they would for k-NN or DBSCAN.
X_scaled = StandardScaler().fit_transform(X)

# Create the model.
# min_samples : the minimum number of points required within a point's
#               neighbourhood for it to be considered a core point —
#               this plays exactly the same role as DBSCAN's min_samples
#               (minPts), and is used to compute core-dist for every point
model = OPTICS(min_samples=5)

# Fit the model. This computes:
#   - core_distances_  : core-dist(o) for every point o
#   - ordering_        : the order in which points were processed
#   - reachability_    : reach-dist for every point, in the ORIGINAL
#                         (unordered) indexing — index it with ordering_
#                         to get the values in processing order, as used
#                         to draw the reachability plot
model.fit(X_scaled)
```

Notice that, unlike `DBSCAN.fit_predict()`, calling `fit()` here does not by itself produce final cluster labels in the way you might expect. `OPTICS` still populates a `labels_` attribute after fitting, but those labels are derived from an automatic cluster-extraction step performed during `fit()`; they are one possible reading of the reachability plot, not the only one. The real output of `fit()` is the reachability structure itself, from which you (or scikit-learn's automatic extraction) can read off clusters at whatever density threshold makes sense for your data.

---

#### Key parameters

| Parameter | Default | What it controls |
|---|---|---|
| `min_samples` | `5` | The minimum number of neighbours (minPts) used to compute core-dist for every point — equivalent to DBSCAN's `min_samples` |
| `max_eps` | `np.inf` | An upper bound on the neighbourhood radius considered when computing reachability distances — leave at the default unless your dataset is very large and you need to cap computation |
| `metric` | `'minkowski'` | Distance metric used to measure neighbourhoods — accepts any metric from Section 5, including `'euclidean'`, `'manhattan'`, and `'cosine'` |
| `cluster_method` | `'xi'` | The automatic cluster-extraction method applied to the reachability plot after fitting — `'xi'` looks for steep up/down slopes in the plot; `'dbscan'` instead applies a single fixed `eps` cutoff, recreating DBSCAN's behaviour from the reachability plot |
| `eps` | `None` | Only used when `cluster_method='dbscan'` — the fixed cutoff applied to the reachability plot, equivalent to drawing a single horizontal line as in Figure 2 |
| `xi` | `0.05` | Only used when `cluster_method='xi'` — controls how steep a rise or fall in the reachability plot must be before it is treated as a cluster boundary; smaller values find more, finer-grained clusters |

The two parameters that matter most in practice are `min_samples`, which shapes the reachability plot itself, and `cluster_method` together with either `xi` or `eps`, which determines how that plot is converted into final cluster labels.

---

#### Reading the output

```python
import numpy as np

# Cluster labels for each point — -1 means noise, exactly as in DBSCAN.
# These come from the automatic extraction step described above.
print(model.labels_)

# core-dist(o) for every point, in the original (unordered) indexing
print(model.core_distances_)

# The order in which OPTICS visited each point — index other _ arrays
# with this to get values in "cluster order", as plotted in Figure 2
print(model.ordering_)

# reach-dist for every point, in the original (unordered) indexing.
# To reproduce the reachability plot from Figure 2:
reachability_ordered = model.reachability_[model.ordering_]

# Count how many clusters were found (excluding noise), and how many
# points were left as noise — same pattern as for DBSCAN
n_clusters = len(set(model.labels_) - {-1})
n_noise    = int((model.labels_ == -1).sum())
print(f'Clusters found: {n_clusters}')
print(f'Noise points:   {n_noise}')
```

The relationship `reachability_[ordering_]` is exactly the array plotted as the bars in Figure 2: `reachability_` gives reach-dist for every point indexed by its original position in `X`, while `ordering_` gives the sequence of original indices that OPTICS visited, so indexing one by the other reorders the values into "cluster order" for plotting.

---

#### Choosing min_samples and the extraction method

As with DBSCAN, `min_samples` is the parameter most worth tuning, and the same guidance from Section 9.1 applies: a value of around 2×(number of features) is a reasonable starting point, with `min_samples=4` or `5` typical for 2D data.

The choice between `cluster_method='xi'` and `cluster_method='dbscan'` is more conceptual than `min_samples`. Setting `cluster_method='dbscan'` with a chosen `eps` reproduces exactly what dragging a single horizontal cutoff line in Figure 2 does: every point with reachability distance above `eps` becomes noise, and the remaining runs become clusters. This is useful when you already have a good `eps` in mind but want OPTICS's improved handling of varying-density clusters at that single threshold.

`cluster_method='xi'`, the default, takes a different approach: rather than a single horizontal cutoff, it scans the reachability plot for points where the reachability distance rises or falls steeply. These steep transitions usually mark the boundary between a dense cluster and a sparser region, and OPTICS uses them to delineate clusters directly. The `xi` parameter controls how steep a transition must be to count as a boundary. This allows `cluster_method='xi'` to find clusters of different densities within the *same* run of `fit()`, which a single `eps` cutoff cannot do.

---

#### Evaluating cluster quality

```python
from sklearn.metrics import silhouette_score

# Filter out noise points before scoring — silhouette score is undefined
# for points with no cluster assignment, exactly as for DBSCAN
mask  = model.labels_ != -1
score = silhouette_score(X_scaled[mask], model.labels_[mask])
print(f'Silhouette score (excluding noise): {score:.3f}')
```

---

> ⚠️ **Warning:** `OPTICS` is more computationally expensive than `DBSCAN`. Computing the full reachability ordering takes longer than DBSCAN's single-pass core/border/noise classification, particularly for large datasets with `algorithm='brute'`. Where possible, leave `algorithm='auto'` so scikit-learn can select a kd-tree or ball-tree, and consider setting `max_eps` to a sensible finite value for very large datasets to bound the neighbourhood search.

**Key takeaway:** OPTICS does not require you to commit to a single $\varepsilon$. The valleys in the reachability plot are clusters, and you can cut at any height to extract clusters at the density level appropriate to your problem.

**When to prefer OPTICS over DBSCAN:** when clusters in your data have noticeably different densities, or when you want to explore the clustering structure at multiple scales.

---

The cell below uses the same small synthetic dataset as before: two well-separated dense clusters of 60 points each, plus 15 scattered noise points. Rather than fitting OPTICS just once, it fits OPTICS independently for every `min_samples` value from 10 to 20, and for each fit records the number of clusters found and the number of points labelled as noise. Plotting both of these against `min_samples` shows how sensitive the result is to this choice, the same systematic sweep used to choose `eps` for DBSCAN in notebook 8, applied here to `min_samples` instead.

In [ ]:
# Listing 5.
# Same dataset as the main example used earlier:
#   - cluster_1, cluster_2 : two well-separated dense blobs of 60 points
#                             each, drawn from tight Gaussians
#   - noise                : 15 points scattered uniformly across the
#                             bounding region, as genuine outliers
rng = np.random.default_rng(42)
cluster_1 = rng.normal(loc=[0, 0], scale=0.3, size=(60, 2))
cluster_2 = rng.normal(loc=[5, 5], scale=0.3, size=(60, 2))
noise     = rng.uniform(low=[-3, -3], high=[8, 8], size=(15, 2))
X = np.vstack([cluster_1, cluster_2, noise])

# Standardise features before fitting — same reasoning as for every other
# distance-based model in this notebook.
X_scaled = StandardScaler().fit_transform(X)

# A small grid of min_samples values, fitted independently, so the effect
# on the resulting clusters can be seen directly on the data. These three
# values were chosen to span the unstable region (10, 15) and the
# start of the plateau identified in the sweep below (20).
min_samples_values = [10, 15, 20]

fig, axes = plt.subplots(1, len(min_samples_values), figsize=(14, 4))
fig.canvas.header_visible = False
fig.canvas.toolbar_visible = False

for ax, ms in zip(axes, min_samples_values):
    # Fit a fresh OPTICS model for this value of min_samples — each panel
    # is an independent fit, not a re-use of any earlier model.
    model = OPTICS(min_samples=ms, cluster_method='xi')
    model.fit(X_scaled)
    labels = model.labels_

    # Plot each cluster label separately so it gets its own colour —
    # same pattern used for DBSCAN's labelled output in Notebook 8.
    for label in sorted(set(labels)):
        mask = labels == label
        if label == -1:
            # Noise points: grey crosses, same convention as Figure 18
            ax.scatter(X[mask, 0], X[mask, 1], marker='x', color='#bbbbbb',
                       s=30, linewidths=1.2)
        else:
            ax.scatter(X[mask, 0], X[mask, 1], s=25, edgecolors='k', lw=0.3)

    # Converting to plain Python ints with int() avoids np.int64 values
    # leaking into the printed title.
    n_clusters = len(set(labels) - {-1})
    n_noise    = int((labels == -1).sum())
    ax.set_title(f'min_samples={ms}\n{n_clusters} clusters, {n_noise} noise', fontsize=9)

    # Axis ticks are removed since the panels are for visual comparison
    # only — the absolute coordinate values aren't important here.
    ax.set_xticks([])
    ax.set_yticks([])

plt.tight_layout()
plt.show()

# ── Sweep across a wider range of min_samples ────────────────────────────────
# Fit OPTICS independently for each min_samples value in this range, and
# record how many clusters and how many noise points each fit produces —
# this is the data the plateau plot below is built from.
min_samples_range = range(5, 31)
n_clusters_list = []
n_noise_list    = []

for ms in min_samples_range:
    sweep_model = OPTICS(min_samples=ms, cluster_method='xi')
    sweep_model.fit(X_scaled)
    sweep_labels = sweep_model.labels_

    n_clusters_list.append(len(set(sweep_labels) - {-1}))
    n_noise_list.append(int((sweep_labels == -1).sum()))

At `min_samples=20`, the picture suddenly becomes what you'd draw by eye: exactly two clusters, one per blob, with only the handful of genuinely scattered points left as noise. 

In practice, you won't usually have the luxury of eyeballing a small-multiples grid like the one above. Real datasets have more than two features, and "what the data actually looks like" isn't something you can just glance at. The systematic approach is to fit OPTICS independently across a range of `min_samples` values, and for each one record the number of clusters found and the number of points labelled as noise.

What you're looking for is a **plateau**: a range of consecutive `min_samples` values that all produce the same number of clusters and a similar (ideally low and stable) noise count. For this dataset, that plateau begins at `min_samples=20`: from there onward, every value produces exactly 2 clusters and 3 noise points. Values below the plateau are unstable; small changes in `min_samples` cause the cluster count to jump around, which is a sign that `xi` is reacting to noise in the data's local density rather than to genuine structure.

Once you've found a plateau, pick a value comfortably inside it rather than right at its edge. The edge of a plateau (here, `min_samples=17` to `19`) can still be sensitive to small changes: at `min_samples=17`, for instance, the cluster count had already settled at 2, but the noise count hadn't yet stabilised. A value a little further into the plateau, like `min_samples=20`, gives you more confidence that the result isn't a coincidence of this particular dataset.

> ⚠️ **Warning:** a stable plateau is a good sign, but it isn't a guarantee of correctness. It only tells you that the *clustering* isn't sensitive to small changes in `min_samples`. If your domain knowledge suggests there should be more or fewer clusters than the plateau shows, that's a signal to look at the data itself, not just to keep adjusting `min_samples` until the count matches what you expected.

The code below plots the plateau directly. The blue line (left axis) shows the number of clusters found for each `min_samples` value; the red line (right axis) shows the number of points labelled as noise. For small `min_samples`, both lines jump around: small changes in `min_samples` produce noticeably different cluster counts and noise counts, which is the instability described above.

From `min_samples=20` onward (shaded green), both lines go flat: every value in this range produces exactly 2 clusters and 3 noise points. This is the plateau, the range of values where the result no longer depends on the exact choice of `min_samples`, which is what makes `min_samples=20` a defensible choice rather than a lucky guess.

In [ ]:
# Listing 6.
# Reuses min_samples_range, n_clusters_list, and n_noise_list computed
# in the cell above — no need to refit anything.
fig, ax1 = plt.subplots(figsize=(8, 5))
fig.canvas.header_visible = False
fig.canvas.toolbar_visible = False

# Cluster count on the left axis, in blue. Plotted against min_samples to
# show how the number of clusters found changes as min_samples increases.
ax1.plot(min_samples_range, n_clusters_list, marker='o', color='steelblue',
         label='Clusters found')
ax1.set_xlabel('min_samples')
ax1.set_ylabel('Number of clusters', color='steelblue')
ax1.tick_params(axis='y', labelcolor='steelblue')

# Noise count gets its own y-axis (ax2), since it's on a much larger
# scale than the cluster count and would otherwise be unreadable on
# the same axis.
ax2 = ax1.twinx()
ax2.plot(min_samples_range, n_noise_list, marker='s', color='tomato',
         label='Noise points')
ax2.set_ylabel('Number of noise points', color='tomato')
ax2.tick_params(axis='y', labelcolor='tomato')

# Shade the plateau region so it's visually obvious where the result
# stabilises — this is the range you'd pick a value from in practice.
# plateau_start=20 was identified by inspecting n_clusters_list and
# n_noise_list: from this point onward both lists hold constant values
# (2 clusters, 3 noise points) for the rest of the range.
plateau_start = 20
ax1.axvspan(plateau_start, max(min_samples_range), color='seagreen', alpha=0.1,
            label='Plateau')

# Annotate the plateau directly on the plot with the values it
# corresponds to, so the reader doesn't have to read them off the axes.
ax1.annotate(
    'Plateau: 2 clusters,\n3 noise points\nfor min_samples ≥ 20',
    xy=(plateau_start, 2), xytext=(plateau_start + 1, 4.5),
    fontsize=9, color='seagreen',
    arrowprops=dict(arrowstyle='->', color='seagreen', lw=1),
)

ax1.set_title('Figure 4: Effect of min_samples on OPTICS results')
ax1.grid(True, alpha=0.15)
plt.tight_layout()
plt.show()



---

# 5. Moving Beyond Clustering: Dimensionality Reduction

🔙 [Back to Table of Contents](#table-of-contents)

So far in unsupervised learning we've asked: *how do I group these data points?* This helps us better understand the structure of our data, or apply labels to data when labels are not easy to obtain. Now we ask something else: *how do I simplify the data?* How do I reduce its dimensionality, making its inherent structure clearer and making tasks such as classification easier?

Real datasets often have dozens, hundreds, or thousands of features. Working with such data causes a variety of problems:

- **The curse of dimensionality** (Notebook 4): distance metrics break down; models need exponentially more data.
- **Visualisation**: we cannot plot more than 2–3 dimensions at once.
- **Redundancy**: many features measure correlated things.
- **Noise**: some features add only noise.

> 📓 **Notebook 4 recap:** Section 6 showed the same underlying problem from three different angles. Geometrically, adding dimensions makes the feature space grow so fast that any fixed set of data points gets spread thinner and thinner. By around 20 dimensions, almost none of your data sits anywhere near the "centre" of the space anymore; it's all out near the edges. This has a direct practical consequence: the amount of data you'd need to keep that space evenly covered by training samples grows just as fast, so the nearest training point to any new example ends up far away in high dimensions. Once your "nearest" neighbour isn't actually close, it stops being a useful guide to what's "similar", and any model based on distance starts to break down. On top of this, every extra feature that carries no real signal, pure noise, nudges all distances upward by roughly the same amount, so points from the same class and points from different classes start to look equally far apart.

This is what Figure 5 below illustrates. It lets you control the number of dimensions, $d$, allowing you to see the three faces of the curse of dimensionality in action. A single slider lets you move $d$ from values of 1 to 20, and three linked panels update as you move it. The annotation strip below summarises the current numbers based on your slider choice.

* The **left panel** shows the volume of a unit hypersphere. This is the higher-dimensional generalisation of a circle's area or a sphere's volume, for a sphere of radius 1. Imagine the hypersphere contained in a cube. This panel shows the volume of the hypersphere as a function of $d$: the curve rises from $d=1$, peaks around $d=5$, and then falls back toward zero. The red marker tracks your current choice of $d$ along this curve. This is the geometric fact underlying everything else in this figure: in low dimensions a sphere fills a healthy fraction of its surrounding cube, but in higher dimensions it shrinks to almost nothing, even though the cube's volume stays fixed at 1.

  > 💡 **Intuition:** picture a $d$-dimensional box spanning $-1$ to $1$ in every direction, with a ball of radius 1 sitting inside it, just touching the middle of every face of the cube. In 1D the "ball" is the whole box: a line segment from $-1$ to $1$. In 2D it's a circle inside a square, and the square's four corners stick out beyond the circle. In 3D it's a sphere in a cube, with even more corner left over. Every extra dimension adds more corners, and a corner is exactly a point that's extreme in *every* direction at once, which is precisely what makes it too far from the centre for a radius-1 ball to reach. The more dimensions, the more of the box becomes "corner", and the ball, stuck at radius 1, gets squeezed into a smaller and smaller sliver in the middle, until it's left occupying almost nothing.

* The **middle panel** is a live simulation. For your chosen $d$, 5,000 points are scattered uniformly inside a $d$-dimensional unit hypercube, and the bar shows what fraction of them fall within $0.5$ of the centre, i.e. how many would count as "near the middle". Earlier dimensions remain shown in blue, so you can watch this fraction collapse as $d$ increases: around 79% in 2D, falling below 1% by around $d=10$.

* The **right panel** shows the number of training samples that would be needed to maintain the same coverage density across the sphere (5 bins per dimension, 10 samples per bin) as $d$ grows. Because this grows as $5^d \times 10$, the y-axis uses a logarithmic scale, with dashed reference lines at one thousand, one million, and one billion samples, so you can see roughly how many dimensions it takes before the requirement becomes practically impossible.

Try moving the slider from $d=1$ up to $d=20$ slowly. By around $d=10$, the simulated fraction in the middle panel has dropped close to zero, and the right panel has already crossed the one-million-samples line: both consequences of the same underlying volume collapse shown on the left. This is the geometry that makes "nearest neighbour" an unreliable concept in high dimensions, and it motivates the dimensionality reduction techniques, starting with PCA, that the rest of this notebook covers.


In [ ]:
# Listing 7.
%matplotlib widget
from visualisations.Figure_5 import show
show()

**Dimensionality reduction** transforms $p$ original features into $k \ll p$ new features that preserve as much important information as possible. Notebook 4 framed feature selection (the process of manually identifying and discarding uninformative features) as one practical response to this problem. Dimensionality reduction is a complementary, more general response: rather than discarding features outright, it constructs a smaller set of *new* features, each a combination of the original ones, chosen so that the new, lower-dimensional representation retains as much of the original information as possible. This is the idea behind **Principal Component Analysis (PCA)**, which we turn to next.

---


## 6. The Building Blocks of PCA

🔙 [Back to Table of Contents](#table-of-contents)

Before we introduce PCA, we need to be sure that we understand three important mathematical tools it relies upon: **variance**, **covariance**, and the **covariance matrix**. 

If you are not comfortable with these, PCA will feel like magic, a black box that somehow finds "the important directions" in your data. Once you understand them, PCA feels like a more obvious solution to a problem.

**You don't have to understand the inner workings if you don't want to.** Just skip ahead if you like. If you think this is too easy, remember that maths backgrounds vary; I'm trying to make this understandable for everyone.

### 6.1 Variance: How Spread Out Is a Variable?

The simplest question we can ask about a single measurement, taken across many examples, is: *how spread out are the values?* If every student scored exactly 60% on an exam, the mean score would be 60% and every student mark would be identical to the mean. In this case there is no spread at all, so the variance is zero. If, on the other hand, scores range all the way from 20% to 95%, the values are scattered widely around the mean, and the variance is larger.

Formally, variance measures the average squared distance from the mean:

$$
\text{Var}(X) = \frac{1}{n-1} \sum_{i=1}^{n} (X_i - \bar{X})^2
$$

Each term $(X_i - \bar{X})$ is one value's distance from the mean: it's positive if $X_i$ is above the mean and negative if it's below. We square these deviations for two reasons: 

1. squaring makes every term positive, so that values above and below the mean both contribute to the spread rather than cancelling each other out.
2. it also penalises larger deviations more heavily than smaller ones, since squaring a big number makes it proportionally bigger still.

In the variance equation above, you might expect to divide by $n$, the number of values, to get an average. But the formula divides by $n-1$ instead. This is known as **Bessel's correction**, and it exists because $\bar{X}$ (the sample mean) itself was *calculated from* the same data: estimating the mean "uses up" one degree of freedom, so dividing by $n$ would systematically underestimate the true variance of the population the sample came from. Dividing by $n-1$ corrects for this and gives an unbiased estimate.

Variance is always a non-negative number, and its units are the *square* of the original variable's units. If $X$ is measured in centimetres, $\text{Var}(X)$ is in centimetres squared. This is why, when we want a measure of spread in the *original* units, we take the square root of the variance, a quantity known as the **standard deviation**.

The code below implements the variance formula from scratch, applies it to two datasets that share the same mean (50) but have very different spreads, and confirms the result matches NumPy's built-in `np.var()`. The two histograms make the difference visible directly: the left dataset clusters tightly around its mean, giving it a small variance, while the right dataset is scattered much more widely around the same mean, giving it a large variance, even though both are centred in exactly the same place.

In [ ]:
# Listing 8.
# Variance from scratch
# ──────────────────────────────────────────────────────────────────────────
# Formula: Var(X) = (1/(n-1)) * sum( (Xi - X_bar)^2 )

# Two datasets with the same mean (50) but very different spreads, so the
# effect of variance is visible directly in the histograms below.
rng_v = np.random.default_rng(1)
tight_data  = rng_v.normal(50,  3, 100)   # small standard deviation -> low variance
spread_data = rng_v.normal(50, 15, 100)   # large standard deviation -> high variance


def my_variance(x):
    """
    Compute sample variance from scratch, following the formula above.

    n - 1 in the denominator is Bessel's correction — see the markdown
    above for why we divide by n - 1 rather than n.
    """
    n = len(x)
    x_bar = sum(x) / n

    # Sum of squared deviations from the mean. Squaring ensures values
    # above and below the mean both contribute positively, rather than
    # cancelling out.
    sum_sq_deviations = sum((xi - x_bar) ** 2 for xi in x)

    return sum_sq_deviations / (n - 1)


var_tight  = my_variance(tight_data)
var_spread = my_variance(spread_data)

# np.var() requires ddof=1 (delta degrees of freedom) to apply the same
# n - 1 correction — without it, NumPy divides by n and gives a slightly
# different (biased) result.
print('Manual variance (tight) :', round(var_tight,  2),
      '  NumPy:', round(np.var(tight_data,  ddof=1), 2))
print('Manual variance (spread):', round(var_spread, 2),
      '  NumPy:', round(np.var(spread_data, ddof=1), 2))

# ── Visualise the two distributions side by side ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.canvas.header_visible = False
fig.canvas.toolbar_visible = False

for ax, data, var, title in [
    (axes[0], tight_data,  var_tight,  f'Small variance ({var_tight:.1f})'),
    (axes[1], spread_data, var_spread, f'Large variance ({var_spread:.1f})'),
]:
    ax.hist(data, bins=25, color='steelblue', edgecolor='white', lw=0.4, alpha=0.85)
    ax.axvline(data.mean(), color='red', lw=2, label=f'Mean = {data.mean():.1f}')
    ax.set_title(title)
    ax.set_xlabel('Value')
    ax.set_ylabel('Count')
    ax.legend(fontsize=9)

plt.suptitle('Figure 6: Variance — how spread out is a single variable?', y=0.98)
plt.tight_layout()
plt.show()

The manual and NumPy results match exactly, confirming the formula is implemented correctly, and the spread-out dataset's variance (223.68) is roughly 34 times larger than the tight dataset's (6.59), reflecting its much wider scatter around the same mean.

---

### 6.2 Covariance: Do Two Variables Move Together?

Variance helps us determine how spread out a single variable is. **Covariance** asks a related question about a *pair* of variables: when one of them is above its mean, is the other typically above its mean too, below its mean, or is there no consistent pattern at all?

> 📓 **Notebook 3 recap:** this question should feel familiar. Section 7.4 of notebook 3 introduced **correlation** to answer exactly this, using the Pearson correlation coefficient $r$. Correlation and covariance are closely related, and the formulas look almost identical, but they differ in one important way, which we'll come back to at the end of this section.

The formula for covariance looks almost identical to variance, but instead of squaring each variable's deviation from its own mean, it multiplies the two variables' deviations together:

$$
\text{Cov}(X, Y) = \frac{1}{n-1} \sum_{i=1}^{n} (X_i - \bar{X})(Y_i - \bar{Y})
$$

The sign of each term in this sum tells you something about that data point. If $X_i$ and $Y_i$ are *both* above their means, or *both* below their means, the two deviations have the same sign, and their product is positive. If one is above its mean while the other is below, the deviations have opposite signs, and their product is negative. Covariance is the average of these products across all $n$ data points, so its overall sign depends on which pattern dominates:

- **Positive covariance** → the two variables tend to be above (or below) their means together: as one increases, the other tends to increase too.
- **Negative covariance** → the two variables tend to move in opposite directions: as one increases, the other tends to decrease.
- **Near-zero covariance** → positive and negative products roughly cancel out: there's no consistent linear relationship between the two variables.

This is the same underlying pattern that Pearson's $r$ was built from in Notebook 3, the numerator of $r$ is precisely $\sum (x_i - \bar{x})(y_i - \bar{y})$, the same sum of products at the heart of covariance. The difference is the denominator: Pearson's $r$ divides this sum by a normalisation factor that rescales the result to always fall between $-1$ and $+1$, regardless of the units $x$ and $y$ are measured in. Covariance skips this normalisation, which means its value depends on the units of $X$ and $Y$: a covariance of $50$ might represent a strong relationship if $X$ and $Y$ are both measured in small numbers, or a weak one if they're measured in the thousands. Correlation is covariance with the units removed; covariance is correlation with the scale of the original data left in.

> ⚠️ **Warning:** notice that $\text{Var}(X) = \text{Cov}(X, X)$: variance is just covariance of a variable with itself, where every deviation is multiplied by itself rather than by a different variable's deviation. This is why the two ideas combine naturally into a single object: the covariance matrix, introduced shortly.

The code below implements the covariance formula from scratch and applies it to three pairs of variables, constructed so that one pair has a positive relationship, one has a negative relationship, and one has no relationship at all. The scatter plots visualise all three cases side by side:

* in the positive case, points cluster along an upward-sloping diagonal: large $X$ values come with large $Y$ values, and small with small.
* In the negative case, the diagonal slopes downward: large $X$ values come with small $Y$ values.
* In the near-zero case, the points form a shapeless cloud with no diagonal trend at all.

The dotted lines through zero on each axis make it easier to see which quadrant each point falls in, and therefore whether its contribution to the covariance sum is positive or negative.

In [ ]:
# Listing 9.
# Covariance from scratch
# ──────────────────────────────────────────────────────────────────────────
# Formula: Cov(X,Y) = (1/(n-1)) * sum( (Xi-X_bar)(Yi-Y_bar) )

def my_covariance(x, y):
    """
    Compute sample covariance from scratch, following the formula above.

    Same n - 1 (Bessel's correction) as my_variance() in Section 6.1 —
    in fact, my_covariance(x, x) would give exactly my_variance(x).
    """
    n = len(x)
    x_bar = sum(x) / n
    y_bar = sum(y) / n

    # Sum of products of paired deviations. zip() pairs up corresponding
    # elements of x and y so each term uses the SAME data point's
    # deviation in both variables.
    sum_products = sum((xi - x_bar) * (yi - y_bar) for xi, yi in zip(x, y))

    return sum_products / (n - 1)


# Three pairs of variables sharing the same x values, constructed to show
# positive, negative, and near-zero covariance:
#   - y_pos  : built from x_base with a positive multiplier (1.5) plus
#              noise — large x tends to come with large y
#   - y_neg  : built from x_base with a negative multiplier (-1.5) plus
#              noise — large x tends to come with small y
#   - y_none : generated completely independently of x_base — no
#              relationship between the two by construction
n = 60
rng_cov = np.random.default_rng(5)
x_base = rng_cov.normal(0, 1, n)
y_pos  =  1.5 * x_base + rng_cov.normal(0, 0.4, n)
y_neg  = -1.5 * x_base + rng_cov.normal(0, 0.4, n)
y_none = rng_cov.normal(0, 1, n)

cov_pos, cov_neg, cov_none = (my_covariance(x_base, y) for y in [y_pos, y_neg, y_none])

print(f'Positive: Cov={cov_pos:+.3f}  |  Negative: Cov={cov_neg:+.3f}  |  None: Cov={cov_none:+.3f}')

# ── Visualise the three relationships ───────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(10, 4))
fig.canvas.header_visible = False
fig.canvas.toolbar_visible = False

for ax, (y, cov, title), col in zip(
    axes,
    [(y_pos,  cov_pos,  'Positive covariance\nBoth move together'),
     (y_neg,  cov_neg,  'Negative covariance\nOne up, other down'),
     (y_none, cov_none, 'Near-zero covariance\nNo consistent pattern')],
    ['steelblue', 'tomato', 'seagreen'],
):
    ax.scatter(x_base, y, color=col, s=40, alpha=0.7, edgecolors='k', lw=0.3)

    # Dotted lines through zero on both axes make it easier to see which
    # quadrant (and therefore which sign of product) each point falls in.
    ax.axhline(0, color='black', lw=0.7, linestyle=':')
    ax.axvline(0, color='black', lw=0.7, linestyle=':')

    ax.set_title(f'{title}\nCov = {cov:+.2f}', fontsize=10)
    ax.set_xlabel('$X$')
    ax.set_ylabel('$Y$')
    ax.grid(True, alpha=0.2)

plt.suptitle('Figure 7: Covariance — direction of relationship between two variables',
             fontsize=12, y=0.98)
plt.tight_layout()
plt.show()

Figure 7 shows three fixed datasets, each illustrating one sign of covariance. Figure 7a lets you explore this more directly: a small set of 8 points sits on a scatter plot, and you can drag any of them around with the mouse.

Two dashed lines mark the current mean of $X$ and the current mean of $Y$, dividing the plot into four quadrants. Every point is coloured according to the sign of its own contribution to $\text{Cov}(X,Y)$, the product $(X_i - \bar{X})(Y_i - \bar{Y})$, so points in the top-right and bottom-left quadrants (where both deviations share the same sign) are blue, and points in the top-left and bottom-right quadrants (where the deviations have opposite signs) are red.

As you drag a point, dotted guide lines show its horizontal and vertical distance from the mean lines: these are exactly $(X_i - \bar{X})$ and $(Y_i - \bar{Y})$, the two quantities being multiplied together for that point. The annotation panel recalculates $\text{Var}(X)$, $\text{Var}(Y)$, and $\text{Cov}(X,Y)$ from scratch after every move, and lists each point's individual contribution to the covariance sum, so you can watch a single point cross from one quadrant to another, see its contribution flip from positive to negative, and see that flip ripple through to the overall $\text{Cov}(X,Y)$ value above it.

Try dragging a point that's currently red (negative contribution) across one of the mean lines into a blue quadrant, and watch $\text{Cov}(X,Y)$ shift accordingly. Then try dragging several points so that they cluster tightly along a diagonal: notice how the covariance grows in magnitude as the points align more consistently with one pattern or the other.

In [ ]:
# Listing 10.
%matplotlib widget
from visualisations.Figure_7a import show
show()

### 6.3 The Covariance Matrix: All Pairwise Relationships at Once

In Notebook 3, Section 7.5, I introduced the **correlation matrix**. This is a square symmetric table, holding the Pearson $r$ value between every pair of features at once, with 1.0 on the diagonal (every feature correlates perfectly with itself) and $r_{ij} = r_{ji}$ in the off-diagonal entries. 

The correlation matrix let you spot redundant features, identify ones likely to be useful for prediction, and detect features that appear unrelated to everything else, all from a single grid, computed once with the numpy function `np.corrcoef`.

The **covariance matrix** $\mathbf{C}$ is built the same way, but with covariances in place of correlations: it is a $p \times p$ table storing every pairwise covariance between $p$ features,

$$
\mathbf{C}_{ij} = \text{Cov}(X_i, X_j)
$$

Just like the correlation matrix, it is symmetric ($\mathbf{C}_{ij} = \mathbf{C}_{ji}$, since $\text{Cov}(X_i, X_j) = \text{Cov}(X_j, X_i)$), and the diagonal entries $\mathbf{C}_{ii} = \text{Var}(X_i)$ hold the variance of each individual feature.

$$
\mathbf{C} =
\begin{bmatrix}
\text{Var}(X_1) & \text{Cov}(X_1, X_2) & \cdots & \text{Cov}(X_1, X_p) \\
\text{Cov}(X_2, X_1) & \text{Var}(X_2) & \cdots & \text{Cov}(X_2, X_p) \\
\vdots & \vdots & \ddots & \vdots \\
\text{Cov}(X_p, X_1) & \text{Cov}(X_p, X_2) & \cdots & \text{Var}(X_p)
\end{bmatrix}
$$

So where the correlation matrix told you about the *direction and strength* of every pairwise relationship on a fixed $-1$ to $+1$ scale, the covariance matrix packages the same pairwise relationships *and* each feature's own spread into one object, in the original units of the data. This combination, every feature's variance along the diagonal and every pairwise covariance off it, is very useful, as you'll see.

---


The code below builds a covariance matrix from scratch for a small dataset of height, weight, and age across eight individuals. It reuses the `my_covariance()` function created in an earlier cell to compute every pairwise entry, including each feature with itself (which gives its variance on the diagonal). The result is checked against the numpy function `np.cov()`, which can compute covariance for us directly. This function expects each row to be a variable and each column an observation, hence the transpose `.T` is needed on `data_3f` to get the data into the right shape.

The heatmap visualises the resulting matrix using the same red-blue colour scheme as the correlation matrix in Notebook 3, with `center=0` so that positive and negative values are immediately distinguishable by colour. Unlike a correlation matrix, however, these values are not bounded between $-1$ and $+1$. They're in the original units of the data (here, roughly centimetres and kilograms squared and cross-products thereof), so their *sizes* aren't directly comparable to one another, only their signs and relative magnitudes within reasonable bounds.

The printed summary at the end picks out three entries to interpret: $\text{Var}(\text{height}) = 66.41$ reflects how spread out the heights are; $\text{Cov}(\text{height}, \text{weight}) = 91.82$ is strongly positive, confirming that taller individuals in this dataset tend to be heavier; and $\text{Cov}(\text{height}, \text{age}) = 13.02$ is positive but much smaller relative to the variances involved, suggesting only a weak relationship between height and age in this small sample.

In [ ]:
# Listing 11.
# A small synthetic dataset with three features for eight individuals.
height = np.array([160, 165, 170, 172, 175, 178, 180, 185], dtype=float)
weight = np.array([ 55,  60,  65,  70,  72,  78,  80,  90], dtype=float)
age    = np.array([ 25,  22,  30,  28,  35,  24,  32,  27], dtype=float)

# np.cov expects rows = variables and columns = observations, so we keep
# data_3f (rows = observations) for general use, but transpose it (.T)
# when passing it to np.cov below.
data_3f = np.column_stack([height, weight, age])
features, names = [height, weight, age], ['height', 'weight', 'age']

# Build the covariance matrix by computing my_covariance() for every pair
# of features, including each feature with itself (which gives its
# variance, on the diagonal).
print('Covariance matrix (manual):')
print(f'  {"":>8}', '  '.join(f'{n:>8}' for n in names))
for xi, ni in zip(features, names):
    row = [my_covariance(xi, xj) for xj in features]
    print(f'  {ni:>8}', '  '.join(f'{v:>8.2f}' for v in row))

# NumPy verification. np.cov expects rows = variables and columns =
# observations — the opposite layout to data_3f — hence the .T.
print('\nNumPy verification (np.cov expects rows = variables, hence .T):')
C = np.cov(data_3f.T)
print(np.round(C, 2))

# ── Visualise as a heatmap ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))
fig.canvas.header_visible = False
fig.canvas.toolbar_visible = False

sns.heatmap(C, annot=True, fmt='.1f', xticklabels=names, yticklabels=names,
            cmap='RdBu_r', center=0, linewidths=0.5, linecolor='white', ax=ax)
ax.set_title('Covariance matrix heatmap\nDiagonal = variance; off-diagonal = covariance')
plt.suptitle('Figure 8: Covariance matrix for height, weight, and age', y=0.98)
plt.tight_layout()
plt.show()

print(f'\nVar(height) = {C[0,0]:.2f}  -> spread of heights')
print(f'Cov(height, weight) = {C[0,1]:.2f}  -> positive: taller tends to be heavier')
print(f'Cov(height, age)    = {C[0,2]:.2f}  -> near zero: not linearly linked here')

---

## 7. Vectors and Matrices

🔙 [Back to Table of Contents](#table-of-contents)

A **vector** is simply an ordered list of numbers, one number per dimension. In two dimensions, a vector such as $\mathbf{v} = [3, 1]$ can be drawn as an arrow starting at the origin $(0, 0)$ and ending at the point $(3, 1)$: it says "go 3 steps right, then 1 step up". In three dimensions a vector would have three numbers, $[v_1, v_2, v_3]$, one for each axis, and so on for any number of dimensions. Beyond 3D we can no longer draw the arrow, only write down the numbers.

This should feel familiar: every data point we've worked with so far, a row of height, weight, and age, for instance, is already a vector. Each feature is one number in the list, and the number of features is the vector's dimension.

An arrow like this has two properties that will matter throughout the rest of this section:

- **Direction**, which way the arrow points. Two vectors can point in the same direction even if they have different lengths; direction is about the *orientation* of the arrow, not its size.
- **Magnitude**, how long the arrow is, i.e. its length. For a vector $\mathbf{v} = [v_1, v_2, \dots, v_p]$, the magnitude is written $\|\mathbf{v}\|$ and calculated as

$$
\|\mathbf{v}\| = \sqrt{\sum_i v_i^2}
$$

This is just the Pythagorean theorem extended to as many dimensions as you like: square each component, add them up, and take the square root. For $\mathbf{v} = [3, 1]$, the magnitude is $\sqrt{3^2 + 1^2} = \sqrt{10} \approx 3.16$.

Figure 9 lets you explore both of these properties directly. A single point is plotted, with a blue arrow drawn from the origin $(0, 0)$ to the point $(3, 1)$; this arrow *is* the vector $\mathbf{v} = [x, y]$. Drag the point anywhere on the plot and watch the arrow follow it.

Two dashed grey lines are drawn alongside the arrow (forming two triangles), with one leg of length $|x|$ (horizontal) and one leg of length $|y|$ (vertical): these are the two quantities inside the square root in the magnitude formula, $\|\mathbf{v}\| = \sqrt{x^2 + y^2}$. The annotation panel on the right shows the calculation worked through step by step for the current vector described by the blue point, along with the angle the arrow makes with the $x$-axis, which describes its direction.

A second, fainter arrow is also shown, always drawn at the same fixed length. This one represents *direction only*: it always points the same way as the blue arrow, but never changes length, regardless of how far you drag the point. Use the two arrows together to build intuition for the distinction between direction and magnitude:

* **Drag the point further along the same line from the origin** (e.g. from $(3, 1)$ out to $(6, 2)$): the blue arrow gets longer and the magnitude in the panel increases, but the faint reference arrow doesn't move at all; its direction hasn't changed, even though the vector's length has.

* **Drag the point to a different angle** (e.g. from $(3, 1)$ to $(1, 3)$): both arrows rotate together. The magnitude may end up similar or even identical, but the direction, and the angle shown in the panel, has changed.

The button labelled "Reset" returns the point to its starting position, $[3, 1]$, the same vector used as a worked example above.

In [ ]:
# Listing 12.
%matplotlib widget
from visualisations.Figure_9 import show
show()

---

You may not realise it, but vectors and matrices can be combined using operations that work a bit like the arithmetic you already know: you can add two vectors together, or multiply a vector by a matrix. We'll work through exactly *how* this multiplication is done, step by step, in a moment. For now, it's worth building some intuition for *what it does*, geometrically, before getting into the mechanics.

Think of a matrix $\mathbf{M}$ as a kind of machine: you feed in a vector $\mathbf{v}$, and it outputs a new vector, $\mathbf{M}\mathbf{v}$. In general, this output vector points in a *different direction* from $\mathbf{v}$, and has a *different magnitude* too. Multiplying by $\mathbf{M}$ effectively grabs every arrow in the space and rotates it, while also stretching or shrinking it, often by different amounts depending on which way the arrow was originally pointing. Two arrows that started out pointing in similar directions might end up pointing in very different directions after being transformed by $\mathbf{M}$.

There is, however, one special case where this transformation behaves in a surprisingly simple way, and it's this special case that turns out to be central to everything in the rest of this notebook. More on that later.

### 7.1 Vector and Matrix Multiplication

Let's make this concrete with small numbers, starting with the simplest case and working up to the "special case" just mentioned.

#### Vector–vector multiplication (the dot product)

Given two vectors $\mathbf{a} = [2, 3]$ and $\mathbf{b} = [4, 1]$, their dot product $\mathbf{a} \cdot \mathbf{b}$ is found by multiplying corresponding entries together and adding the results:

$$
\mathbf{a} \cdot \mathbf{b} = (2 \times 4) + (3 \times 1) = 8 + 3 = 11
$$

The result is a single number, not a vector. 

#### Matrix–vector multiplication

Now consider the matrix

$$
\mathbf{M} = \begin{bmatrix} 2 & 1 \\ 1 & 2 \end{bmatrix},
$$

and the vector $\mathbf{v} = [1, 1]$. To compute $\mathbf{M}\mathbf{v}$, we take the dot product of *each row* of $\mathbf{M}$ with $\mathbf{v}$, and stack the results into a new vector:

$$
\mathbf{M}\mathbf{v} = \begin{bmatrix} 2 & 1 \\ 1 & 2 \end{bmatrix} \begin{bmatrix} 1 \\ 1 \end{bmatrix} = \begin{bmatrix} (2 \times 1) + (1 \times 1) \\ (1 \times 1) + (2 \times 1) \end{bmatrix} = \begin{bmatrix} 3 \\ 3 \end{bmatrix}
$$

So $\mathbf{M}$ has transformed $\mathbf{v}$ into a new vector $[3, 3]$. Notice that $[3, 3]$ points in *exactly the same direction* as $[1, 1]$, it's just three times as long. Try this in Figure 9 to confirm it: move from $[1, 1]$ to $[3, 3]$ and note that the direction doesn't change.

Now compare this to what happens with a different starting vector, say $\mathbf{v}_2 = [3, 1]$, the same vector used as an example earlier in this section:

$$
\mathbf{M}\mathbf{v}_2 = \begin{bmatrix} 2 & 1 \\ 1 & 2 \end{bmatrix} \begin{bmatrix} 3 \\ 1 \end{bmatrix} = \begin{bmatrix} (2 \times 3) + (1 \times 1) \\ (1 \times 3) + (2 \times 1) \end{bmatrix} = \begin{bmatrix} 7 \\ 5 \end{bmatrix}
$$

This time, $[7, 5]$ points in a noticeably *different* direction from $[3, 1]$: multiplying by $\mathbf{M}$ has rotated this vector as well as scaled it.

So the result of $\mathbf{v}=[1, 1]$ when multiplied by $\mathbf{M}$ pointed exactly where it started, only three times longer, while $\mathbf{v}=[3, 1]$ when multiplied by $\mathbf{M}$ ended up pointing somewhere completely different. 

Vectors that behave like $[1, 1]$, staying on the same line through the origin and only being stretched or shrunk, are called **eigenvectors** of $\mathbf{M}$. The factor by which they're stretched (here, $3$) is called the corresponding **eigenvalue**. The vector $[3, 1]$, which got rotated, is not an eigenvector of $\mathbf{M}$. 

Every matrix has its own particular set of eigenvectors, directions it leaves unchanged, only scaling them, and a corresponding eigenvalue for each one, telling you exactly how much that direction gets stretched or shrunk. Finding these special directions for a particular matrix is what the next section is about.

---

Figure 10 lets you experiment with matrix-vector multiplication directly, using the same matrix $\mathbf{M} = \begin{bmatrix} 2 & 1 \\ 1 & 2 \end{bmatrix}$ from the example above. The left panel shows your chosen vector $\mathbf{v} = [x, y]$ as an arrow from the origin. Drag the point, or type exact values into the $x$ and $y$ boxes above the plot, and the right panel updates to show $\mathbf{M}\mathbf{v}$, the result of multiplying $\mathbf{v}$ by $\mathbf{M}$. The annotation panel works through the multiplication step by step for whatever vector you've chosen, exactly as in the worked examples.

The annotation panel also tells you whether your current $\mathbf{v}$ is an eigenvector of $\mathbf{M}$. In the worked examples, $[1, 1]$ came out *exactly* unchanged in direction, but if you try to find this by dragging with a mouse, you'll never land on $[1, 1]$ *exactly*; you might get $[1.02, 0.97]$, which is extremely close but not quite identical. Strictly speaking, only $[1,1]$ itself, and other vectors lying exactly on that same line through the origin (like $[2,2]$ or $[-3,-3]$), are true eigenvectors of $\mathbf{M}$; every other vector, however close, technically isn't.

This is why the figure checks whether $\mathbf{v}$ is **approximately** an eigenvector, by measuring the angle between $\mathbf{v}$ and $\mathbf{M}\mathbf{v}$. If that angle is very close to $0°$ (pointing the same way) or very close to $180°$ (pointing exactly opposite), both arrows turn gold and the panel reports an estimated eigenvalue, the factor by which $\mathbf{v}$'s length changed. This is a practical concession to the fact that you're dragging a point with a mouse: it lets you *find* the eigenvector directions by feel, without needing pixel-perfect precision, while still being strict enough that an arbitrary vector like $[3, 1]$ won't accidentally trigger it.

Try dragging $\mathbf{v}$ slowly around the origin in a full circle. For almost every position, $\mathbf{M}\mathbf{v}$ points somewhere visibly different from $\mathbf{v}$, but at two specific angles (and their exact opposites), both arrows will suddenly align and turn gold. These two directions are $\mathbf{M}$'s eigenvectors, and the eigenvalue shown at each tells you how strongly $\mathbf{M}$ stretches vectors pointing that way.


In [ ]:
# Listing 13.
%matplotlib widget
from visualisations.Figure_10 import show
show()

To summarise: a **vector** is just an ordered list of numbers. In two or three dimensions you can picture this as an arrow, and every arrow has a **direction** (which way it points) and a **magnitude** (how long it is). A **matrix**, when multiplied by a vector, transforms that arrow, generally rotating it to point somewhere new, while also stretching or shrinking it.

An **eigenvector** of a matrix $\mathbf{M}$ is one of the special directions that $\mathbf{M}$ does *not* rotate: multiplying by $\mathbf{M}$ only changes its length (possibly flipping it to point exactly the opposite way), never its direction. The factor by which its length changes is called the corresponding **eigenvalue**. Figure 10 let you search for these directions directly: for almost every vector you tried, $\mathbf{M}\mathbf{v}$ pointed somewhere different from $\mathbf{v}$, but at two specific directions (and their opposites), $\mathbf{v}$ and $\mathbf{M}\mathbf{v}$ lined up; those are $\mathbf{M}$'s eigenvectors.

Why does any of this matter for machine learning? Because, as we're about to see, the covariance matrix from Section 6.3 is itself just a matrix, and *its* eigenvectors turn out to point in exactly the directions along which your data varies the most and the least. Finding those directions is the entire idea behind PCA.


---

## 8. Eigenvectors and Eigenvalues

🔙 [Back to Table of Contents](#table-of-contents)

Section 7 showed, by example, that most vectors get rotated when multiplied by a matrix, but a special few don't. Multiplying $\mathbf{M} = \begin{bmatrix} 2 & 1 \\ 1 & 2 \end{bmatrix}$ by $[1,1]$ produced $[3,3]$: the same direction, three times the length. We called vectors that do this when multiplied by a matrix **eigenvectors** of the matrix. More formally:

A non-zero vector $\mathbf{v}$ is an **eigenvector** of a matrix $\mathbf{C}$ if multiplying by $\mathbf{C}$ leaves its direction unchanged: the result is just $\mathbf{v}$ scaled by some number $\lambda$ (lambda), called the **eigenvalue**:

$$
\mathbf{C} \, \mathbf{v} = \lambda \, \mathbf{v}
$$

This is exactly the relationship Figure 10 let you search for. Dragging $\mathbf{v}$ around the origin, you would have found that at $[1,1]$ (and along the same line through the origin), $\mathbf{M}\mathbf{v}$ pointed in exactly the same direction as $\mathbf{v}$, just 3 times longer, so $[1,1]$ is an eigenvector of $\mathbf{M}$ with eigenvalue $\lambda = 3$. 

At $[1,-1]$, $\mathbf{M}\mathbf{v}$ again pointed in exactly the same direction, this time with no change in length at all, so $[1,-1]$ is also an eigenvector, with eigenvalue $\lambda = 1$.

So far, $\mathbf{C}$ has just been some matrix we picked to demonstrate an idea. But $\mathbf{C}$ can be *any* square matrix, even the covariance matrix from Section 6.3. When it is, the eigenvectors and eigenvalues stop being abstract and start describing your dataset directly:

- Each **eigenvector** is a direction in feature space, a direction the data actually varies along.
- Its **eigenvalue** tells you *how much* the data varies along that direction: a large eigenvalue means the data is spread out widely along that eigenvector; a small eigenvalue means the data barely changes along it.
- For a covariance matrix specifically, the eigenvectors are always **orthogonal**, at right angles to each other. With two features, this gives exactly two eigenvectors, perpendicular to one another, like a pair of axes rotated to line up with the natural spread of the data.

In other words: find the eigenvectors of a dataset's covariance matrix, and you've found the directions along which the data spreads out most and least, ranked by how much variance each one captures. This is the foundation PCA is built on. The goal then is to find the eigenvectors.


---

### 8.1 Where Do Eigenvalues Come From? A Walkthrough

So far we've taken eigenvalues and eigenvectors as given, but where do these numbers actually come from? After all, we'd need to find them for our covariance matrix to discover the direction in which our data varies and how strong that variation is. For a $2 \times 2$ matrix, the calculation is short enough to do by hand, and working through it removes any sense that eigenvalues are something mysterious.

Let's say we have a covariance matrix $\mathbf{C} = \begin{bmatrix} 2 & 1 \\ 1 & 2 \end{bmatrix}$, and we want to find its eigenvectors (the directions) and eigenvalues (the stretch factors $\lambda$ along those directions). We don't yet know what these are; they exist, somewhere, but right now they're unknowns we need to solve for, just like $x$ in any other algebra problem.

**Step 1: Write down what a valid eigenvector must satisfy.** From Section 8, an eigenvector $\mathbf{v} = [x, y]$ and its eigenvalue $\lambda$ must satisfy $\mathbf{C}\mathbf{v} = \lambda\mathbf{v}$. Substituting in our actual matrix:

$$
\begin{bmatrix} 2 & 1 \\ 1 & 2 \end{bmatrix} \begin{bmatrix} x \\ y \end{bmatrix} = \lambda \begin{bmatrix} x \\ y \end{bmatrix}
$$

We have three unknowns here ($x$, $y$, and $\lambda$) and one matrix equation. Let's turn this into something we can actually work with algebraically.

**Step 2: Carry out the matrix-vector multiplication on the left.** Using the row-by-row rule from Section 7.1 (dot product of each row of the matrix with $\mathbf{v}$):

$$
\text{row 1: } (2 \times x) + (1 \times y) = 2x + y
$$

$$
\text{row 2: } (1 \times x) + (2 \times y) = x + 2y
$$

So $\mathbf{C}\mathbf{v} = \begin{bmatrix} 2x + y \\ x + 2y \end{bmatrix}$. This means we have part of $\mathbf{C}\mathbf{v} = \lambda\mathbf{v}$ that we know we must satisfy. But what about the $\lambda\mathbf{v}$ part?

**Step 3: Carry out the multiplication on the right.** $\lambda\mathbf{v} = \lambda \begin{bmatrix} x \\ y \end{bmatrix} = \begin{bmatrix} \lambda x \\ \lambda y \end{bmatrix}$, multiplying a vector by the scalar $\lambda$ just multiplies each entry by $\lambda$. Now we know the values of both sides.

**Step 4: Set the two sides equal, entry by entry.** 

We now know that $\mathbf{C}\mathbf{v} = \lambda\mathbf{v}$ becomes $\begin{bmatrix} 2x + y \\ x + 2y \end{bmatrix} = \begin{bmatrix} \lambda x \\ \lambda y \end{bmatrix}$. 

We also know that two vectors are equal only if their corresponding entries are equal, so this gives us two separate equations:

$$
2x + y = \lambda x \qquad \text{and} \qquad x + 2y = \lambda y
$$

**Step 5: Move everything to one side.** 

We want each equation to equal zero, so we can later ask "for which $\lambda$ does a non-zero solution exist?" Subtract $\lambda x$ from both sides of the first equation, and $\lambda y$ from both sides of the second:

$$
2x + y - \lambda x = 0 \qquad \text{and} \qquad x + 2y - \lambda y = 0
$$

**Step 6: Group the like terms.** 

In the first equation, $2x$ and $-\lambda x$ both involve $x$, so we can combine them into $(2 - \lambda)x$. In the second equation, $2y$ and $-\lambda y$ both involve $y$, combining into $(2-\lambda)y$:

$$
(2-\lambda)x + y = 0 \qquad \text{and} \qquad x + (2-\lambda)y = 0
$$

These are two equations describing how $x$ and $y$ must relate to each other, for a given value of $\lambda$. Both equations must be true *at the same time*, for the *same* $x$ and $y$: together, they're the rule a valid eigenvector must obey.

**Step 7: Notice the obvious solution, and why it doesn't help.** 

If $x = 0$ and $y = 0$, both equations are satisfied, for *any* value of $\lambda$: $(2-\lambda)(0) + 0 = 0$ and $0 + (2-\lambda)(0) = 0$ are both true no matter what $\lambda$ is. But $\mathbf{v} = [0, 0]$ isn't an eigenvector; eigenvectors are defined to be non-zero. So this "solution" tells us nothing about $\lambda$. The real question is: **for which values of $\lambda$ can $x$ and $y$ be non-zero and still satisfy both equations?**

**Step 8: See what happens for an arbitrary $\lambda$.** 

Before introducing any new machinery, let's just try a value and see what goes wrong. Suppose $\lambda = 2$. Substituting into our two equations:

$$
(2-2)x + y = 0 \implies y = 0
$$

$$
x + (2-2)y = 0 \implies x = 0
$$

The first equation immediately tells us $y = 0$, and the second tells us $x = 0$. So for $\lambda = 2$, the *only* solution is $x = 0, y = 0$, the trivial one we just ruled out. $\lambda = 2$ is not an eigenvalue of $\mathbf{C}$.

What went wrong? Each equation, on its own, describes a straight line of $(x, y)$ points satisfying it. For $\lambda=2$, the first equation $y=0$ is the line "$y$ equals zero" (the $x$-axis), and the second equation $x=0$ is the line "$x$ equals zero" (the $y$-axis). These are two *different* lines, and the only point where two different lines through the origin can meet is the origin itself, $(0,0)$. That's why we're stuck with only the trivial solution.

For a non-zero solution to exist, both equations need to describe the *same* line, so that every point satisfying one equation automatically satisfies the other too, giving infinitely many shared $(x,y)$ pairs, not just $(0,0)$.

**Step 9: Find a way to check whether the two lines coincide, without finding the lines.** 

Checking "do these two lines coincide?" by working out each line's slope and comparing would work, but there's a quicker tool: the **determinant**. For any $2\times2$ matrix $\mathbf{A} = \begin{bmatrix} a & b \\ c & d \end{bmatrix}$, its determinant is the single number:

$$
\det(\mathbf{A}) = ad - bc
$$

Multiply the two diagonal entries together, multiply the two off-diagonal entries together, and subtract the second product from the first. The key fact we'll use is: **the two-equation system $a x + by = 0$ and $cx + dy = 0$ has a non-zero solution for $(x,y)$ exactly when $\det(\mathbf{A}) = 0$.** We saw this concretely in Step 8: for $\lambda=2$, our coefficient matrix would have been $\begin{bmatrix} 0 & 1 \\ 1 & 0 \end{bmatrix}$, with determinant $(0)(0) - (1)(1) = -1 \neq 0$, correctly signalling "only the trivial solution exists". If instead the determinant comes out to $0$, the two equations turn out to describe the same line, and non-zero solutions exist along it.

**Step 10: Build the coefficient matrix for our equations.** 

Look back at Step 6: $(2-\lambda)x + y = 0$ and $x + (2-\lambda)y = 0$. The coefficient matrix collects the numbers multiplying $x$ and $y$ in each equation: row 1 from the first equation, row 2 from the second. In the first equation, $x$ has coefficient $(2-\lambda)$ and $y$ has coefficient $1$ (since $y$ appears on its own, with an implicit coefficient of 1). In the second equation, $x$ has coefficient $1$ and $y$ has coefficient $(2-\lambda)$. This gives:

$$
\begin{bmatrix} 2-\lambda & 1 \\ 1 & 2-\lambda \end{bmatrix}
$$

**Step 11: Compute the determinant of this matrix.** 

Using $\det(\mathbf{A}) = ad-bc$ from Step 9, with $a=d=(2-\lambda)$ and $b=c=1$:

$$
\det = (2-\lambda)(2-\lambda) - (1)(1) = (2-\lambda)^2 - 1
$$

**Step 12: Set the determinant to zero.** 

By Step 9, non-zero solutions for $(x,y)$ exist exactly when this determinant is zero:

$$
(2-\lambda)^2 - 1 = 0
$$

**Step 13: Solve for $\lambda$.** 

Add $1$ to both sides:

$$
(2-\lambda)^2 = 1
$$

Take the square root of both sides. Remember that a square root has two possible signs: $1$ could have come from squaring $+1$ or $-1$:

$$
2 - \lambda = \pm 1
$$

This gives two cases. If $2 - \lambda = 1$, then $\lambda = 1$. If $2 - \lambda = -1$, then $\lambda = 3$. So:

$$
\lambda = 1 \qquad \text{or} \qquad \lambda = 3
$$

These are the only two values of $\lambda$ for which $x$ and $y$ can be non-zero while still satisfying both equations from Step 6, in other words, the only two eigenvalues of $\mathbf{C}$.

**Step 14: Find the eigenvector for $\lambda = 1$.** 

Substitute $\lambda=1$ into either equation from Step 6, using the first, $(2-\lambda)x + y = 0$:

$$
(2-1)x + y = 0 \implies (1)x + y = 0 \implies x + y = 0 \implies y = -x
$$

So for $\lambda=1$, any vector where $y=-x$ is an eigenvector: $[1,-1]$, $[2,-2]$, $[3,-3]$, or generally $[n, -n]$ for any non-zero $n$. (As a check, the second equation from Step 6 gives the same condition: $x + (2-1)y = 0 \implies x+y=0 \implies y=-x$, both equations describe the same line, exactly as Step 8 said they must.)

**Step 15: Find the eigenvector for $\lambda = 3$.** 

Substitute $\lambda=3$ into the first equation from Step 6:

$$
(2-3)x + y = 0 \implies (-1)x + y = 0 \implies -x+y=0 \implies y=x
$$

So for $\lambda=3$, any vector where $y=x$ is an eigenvector: $[1,1]$, $[2,2]$, $[3,3]$, or generally $[n,n]$ for any non-zero $n$, the direction that Figure 10 highlighted in gold. An **eigenvalue** is the number $\lambda$ that pairs with an eigenvector $\mathbf{v}$ in the equation $\mathbf{C}\mathbf{v} = \lambda\mathbf{v}$: it's the factor by which $\mathbf{C}$ stretches (or shrinks) $\mathbf{v}$, without changing its direction. When $\mathbf{C}$ is a covariance matrix, this stretch factor has a concrete meaning: it's the amount of variance the data has along the direction that $\mathbf{v}$ points in. A large eigenvalue means the data is spread out widely along that direction; a small eigenvalue (close to zero) means the data barely varies along it at all. Every eigenvector comes with its own eigenvalue, and together they tell you not just *which* directions matter in your data, but *how much* each one matters.

This is the full picture: $\mathbf{C} = \begin{bmatrix} 2 & 1 \\ 1 & 2 \end{bmatrix}$ has exactly two eigenvalue/eigenvector pairs, $\lambda=1$ with eigenvectors along $y=-x$, and $\lambda=3$ with eigenvectors along $y=x$, found entirely through ordinary algebra, with the determinant appearing only as a shortcut for checking whether two lines coincide.

**RESULT**

What this walkthrough has actually shown is how to compute the eigenvectors and eigenvalues for any matrix, including a correlation matrix. In practice you'd let the code handle it (`np.linalg.eig` or `np.linalg.eigh`), but the same process underlies it: finding the directions (eigenvectors) in which your data varies the most, and the strength of that variation (eigenvalues).

The table below summarises the steps above.

| Step | What we do | Result |
|---|---|---|
| 1–4 | Substitute $\mathbf{C}$ into $\mathbf{C}\mathbf{v}=\lambda\mathbf{v}$, multiply out both sides, and set entries equal | $2x+y=\lambda x$ and $x+2y=\lambda y$ |
| 5–6 | Move everything to one side and group like terms | $(2-\lambda)x+y=0$ and $x+(2-\lambda)y=0$ |
| 7–8 | Rule out $x=y=0$; check an arbitrary $\lambda$ (e.g. $\lambda=2$) to see only the trivial solution exists | Need a different test for which $\lambda$ work |
| 9 | Introduce the determinant as a test for whether the two equations describe the same line | $\det\begin{bmatrix}a&b\\c&d\end{bmatrix}=ad-bc=0 \iff$ non-zero solutions exist |
| 10–11 | Build the coefficient matrix and compute its determinant | $\det = (2-\lambda)^2-1$ |
| 12–13 | Set determinant to zero and solve for $\lambda$ | $\lambda=1$ or $\lambda=3$ |
| 14 | Substitute $\lambda=1$ to find its eigenvector direction | $y=-x$ |
| 15 | Substitute $\lambda=3$ to find its eigenvector direction | $y=x$ |

---

Let's step back and tie all of this together. A **vector** is an arrow with a **direction** and a **magnitude**, and crucially, every data point in your dataset is a vector, with as many entries as there are features. **Variance** told us how spread out a single feature is, and **covariance** told us how a pair of features vary *together*. The **covariance matrix** packaged all of this up into a single object: every feature's own variance along the diagonal, and every pairwise covariance off it, a complete numerical summary of how your data spreads out in feature space.

What we didn't have, until now, was a way to turn that summary into *directions*. The covariance matrix tells us, in terms of the original features, how the data varies, but "height varies with weight" isn't itself a direction you can point to. This is exactly the gap eigenvectors fill. Section 8.1 showed, step by (painful?) step, how to take a matrix like $\mathbf{C} = \begin{bmatrix} 2 & 1 \\ 1 & 2 \end{bmatrix}$ and extract from it a small number of special directions, its eigenvectors, each with an eigenvalue describing how much "stretch" lies along it.

When $\mathbf{C}$ is a covariance matrix, those directions and stretches stop being abstract: each eigenvector is a direction through feature space, and its eigenvalue is the amount of variance the data has along that direction. The eigenvector with the largest eigenvalue points in the direction the data spreads out *most*, the direction along which individual data points differ from each other the most. The eigenvector with the smallest eigenvalue points in the direction the data barely varies at all, a direction where almost every data point looks the same.

This is the missing piece, and it's exactly what **Principal Component Analysis (PCA)** is. PCA takes a dataset's covariance matrix, computes its eigenvectors and eigenvalues, and uses the eigenvectors as a brand new set of axes for the data, called **principal components**, ordered by their eigenvalues from largest to smallest. The first principal component is the single direction along which the data varies the most; the second is the direction (at right angles to the first) along which it varies next most; and so on.

Because these new axes are *ranked* by how much variance each one captures, PCA gives you something the original features never could: a way to ask "how many of these new axes do I actually need to keep?" If the first two or three principal components capture almost all of the variance in a dataset with twenty original features, you can describe that dataset using just those two or three numbers per data point instead of twenty, with barely any information lost. This is dimensionality reduction in action, and the rest of this notebook is about how to do it, what it looks like, and how to decide how many components to keep.

---

Figure 11 takes everything from Sections 6 to 8 and shows it on a single, real dataset. The scatter cloud on the left is 200 points with two correlated features. You can see by eye that they tend to increase together, the same kind of positive relationship discussed in Section 6.2. The black cross marks the mean of the data, $(\bar{x}, \bar{y})$, the point all of the eigenvector arrows are drawn from.

The covariance matrix $\mathbf{C}$ for this dataset is shown in the annotation panel, computed exactly as in Section 6.3: its diagonal entries are the variances of each feature, and its off-diagonal entries are their covariance. Its eigenvectors and eigenvalues, computed using `np.linalg.eigh` (the version of `np.linalg.eig` suited to symmetric matrices like covariance matrices), are drawn directly on the scatter plot as two double-headed arrows, both passing through the mean.

The red arrow, **PC1**, points along the eigenvector with the *largest* eigenvalue, and visually, it runs straight along the long axis of the data cloud, the direction in which the points are most spread out. The green arrow, **PC2**, points along the eigenvector with the *smallest* eigenvalue, and is always perpendicular to PC1, exactly as Section 8 said covariance matrix eigenvectors must be. Notice how short PC2 is compared to PC1: the data barely varies at all in this direction.

Each arrow's length is set to $2\sqrt{\lambda}$, two standard deviations along that direction, so the length of each arrow is a direct visual representation of its eigenvalue, and therefore of how much of the data's total variance lies along it. The annotation panel makes this precise: PC1's eigenvalue of $4.44$ accounts for about $95\%$ of the total variance in this dataset, while PC2's eigenvalue of $0.23$ accounts for the remaining $5\%$. The "total variance" itself is just the sum of the two eigenvalues, which, as the panel notes, is the same as the sum of the two diagonal entries of $\mathbf{C}$, i.e. $\text{Var}(\text{Feature 1}) + \text{Var}(\text{Feature 2})$.

This is the geometric picture behind PCA: instead of describing each point using its original Feature 1 and Feature 2 coordinates, you could instead describe it using how far along PC1 and PC2 it sits. Since PC2 captures only 5% of the variance, you could even drop it almost entirely, describing each point with a single number (its position along PC1) and losing very little information. That's dimensionality reduction, and it's exactly what we'll do next.

In [ ]:
# Listing 14.
%matplotlib widget
from visualisations.Figure_11 import show
show()

---

## 9. Principal Component Analysis (PCA)

🔙 [Back to Table of Contents](#table-of-contents)

We now have every ingredient:

| Ingredient | What it gives us |
|---|---|
| Variance | Spread of each individual feature |
| Covariance | Relationship between pairs of features |
| Covariance matrix | All pairwise relationships in one object |
| Eigenvectors | The directions of maximum spread |
| Eigenvalues | How much spread there is along each direction |

**PCA combines them all.** It computes the covariance matrix of your data, finds its eigenvectors and eigenvalues, and uses the eigenvectors as a new set of axes, the **principal components**, ranked by their eigenvalues from largest to smallest. The first principal component, PC1, is the single direction along which your data varies the most; PC2 is the direction (always perpendicular to PC1) along which it varies next most; and so on, for as many principal components as you have original features.

Once you have these axes, PCA can be put to use in two related but distinct ways, and it's worth being clear about the difference up front.

**Use 1: Reducing the number of dimensions.** This is the use case the rest of this notebook focuses on. If the first $k$ principal components together capture almost all of the variance in your dataset, as PC1 did in Figure 11, capturing 95% on its own, then describing each data point using only its position along those $k$ axes loses very little information, even if your original data had many more than $k$ features. Crucially, this isn't just a *description* of the data, it's a **transformation** of it. PCA produces a new dataset, with $k$ columns instead of $p$, where each column is a new feature (a principal component) rather than one of your original measurements. From this point on, you work with this *transformed* dataset: if you're building a classifier, clustering the data, or visualising it, you do so using these $k$ new coordinates, not the original $p$ features. The new dataset is smaller, but it's also a genuinely different dataset, its columns no longer correspond to "height" or "weight" or whatever your original features were, but to combinations of them.

**Use 2: Understanding which original features matter most.** Separately from reducing dimensions, PCA can help you understand your *original* features better. Each eigenvector is a list of numbers, one per original feature. These numbers are called **loadings**, and they tell you how much each original feature contributes to that principal component's direction. A feature with a large loading on PC1 is one that varies a lot in the direction PC1 points, in other words, a feature that's strongly involved in whatever the dominant pattern of variation in your dataset is. A feature with small loadings across every principal component that captures meaningful variance is one that barely varies at all relative to everything else, and might be a candidate for removal entirely. This use of PCA isn't about producing a smaller dataset to feed into another model, it's a diagnostic tool, used to understand the structure of your data before deciding what to do next.

Both uses rely on exactly the same calculation; only what you do with the result differs. The algorithm below produces the principal components; what follows afterwards is choosing how many of them to keep (Use 1) or examining their loadings (Use 2).

### 9.1 The Full Algorithm

1. **Standardise** the data (subtract mean, divide by standard deviation): same reasoning as Section 5.1's discussion of distance-based methods. Without this, features measured on larger scales would dominate the covariance matrix regardless of how informative they actually are.
2. **Compute the covariance matrix** $\mathbf{C}$ of the standardised data, as in Section 6.3.
3. **Compute eigenvectors and eigenvalues** of $\mathbf{C}$, as in Section 8.
4. **Sort** the eigenvector/eigenvalue pairs by descending eigenvalue, so the first pair corresponds to PC1, the direction of greatest variance, the second to PC2, and so on.
5. **Select** the top $k$ eigenvectors, however many principal components you want to keep, and arrange them as columns of a matrix $\mathbf{W}$, with shape $p \times k$ ($p$ original features, $k$ chosen components). This is called the **projection matrix**.
6. **Project** the standardised data onto these new axes by multiplying:

$$
\mathbf{X}_{\text{PCA}} = \mathbf{X}_{\text{std}} \cdot \mathbf{W}
$$

To see what this multiplication actually does, it helps to think back to Section 7: there, a matrix $\mathbf{M}$ multiplied by a single vector $\mathbf{v}$ produced a new vector $\mathbf{M}\mathbf{v}$, the same vector, transformed into new coordinates. Here, $\mathbf{X}_{\text{std}}$ is your *entire dataset*, written as a matrix with shape $n \times p$ ($n$ data points, each a row, with $p$ original features as columns). 

Multiplying it by $\mathbf{W}$, the matrix of eigenvectors, with shape $p \times k$, does the *same thing to every row at once*: each row of $\mathbf{X}_{\text{std}}$ (one data point, as a vector of $p$ features) is transformed into a new row of $k$ numbers, its coordinates along the $k$ principal components. The matrix multiplication rule from Section 7.1, dot product of each row with each column, applies exactly as before, just repeated across all $n$ rows in a single operation.

The result, $\mathbf{X}_{\text{PCA}}$, has shape $n \times k$: the same $n$ data points, but now described using $k$ new coordinates instead of $p$ original ones. These new coordinates are called the **scores**, and $\mathbf{X}_{\text{PCA}}$ is the transformed dataset referred to in Use 1 above, the one you'd go on to use for classification, clustering, or visualisation.

To be completely clear: the matrix multiplication itself is what transforms the input data, reducing its dimensionality in the process. The worked example below makes this concrete with small numbers.

---

#### Example: Dimensionality Reduction via Matrix Multiplication

Suppose our standardised input dataset $\mathbf{X}_{\text{std}}$ has 3 samples and 3 features (a $3 \times 3$ matrix, each row a data point and each column a feature), and $\mathbf{W}$ holds the top 2 eigenvectors of its covariance matrix (a $3 \times 2$ matrix, each column an eigenvector, representing a direction of maximum variance in the data), selected to project the data onto a lower-dimensional space:

$$
\mathbf{X}_{\text{std}} = \begin{bmatrix}
1 & 2 & 3 \\
4 & 5 & 6 \\
7 & 8 & 9 \\
\end{bmatrix}
\qquad
\mathbf{W} = \begin{bmatrix}
1 & 0 \\
0 & 1 \\
0 & 0 \\
\end{bmatrix}
$$

Following Step 6 above, $\mathbf{X}_{\text{PCA}} = \mathbf{X}_{\text{std}} \cdot \mathbf{W}$, computed the same row-dot-column way as every matrix multiplication in Section 7.1:

$$
\mathbf{X}_{\text{PCA}} = \mathbf{X}_{\text{std}} \cdot \mathbf{W} =
\begin{bmatrix}
(1 \cdot 1 + 2 \cdot 0 + 3 \cdot 0) & (1 \cdot 0 + 2 \cdot 1 + 3 \cdot 0) \\
(4 \cdot 1 + 5 \cdot 0 + 6 \cdot 0) & (4 \cdot 0 + 5 \cdot 1 + 6 \cdot 0) \\
(7 \cdot 1 + 8 \cdot 0 + 9 \cdot 0) & (7 \cdot 0 + 8 \cdot 1 + 9 \cdot 0) \\
\end{bmatrix}
=
\begin{bmatrix}
1 & 2 \\
4 & 5 \\
7 & 8 \\
\end{bmatrix}
$$

Each row of this output is one of our original data points, now described in just 2 coordinates instead of 3, the reduced dataset in the new space defined by the two principal components.

## 10. PCA with scikit-learn

🔙 [Back to Table of Contents](#table-of-contents)

The six-step algorithm above can be implemented entirely with NumPy. But in practice, scikit-learn's `PCA` class handles all six steps for you, in an interface that mirrors `DBSCAN` and `OPTICS` from Notebook 8 and earlier in this notebook: a model object, a `fit()` step, and attributes exposing the results.

---

#### Creating and fitting the model

```python
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Standardise features first — Step 1 of the algorithm above, and the
# same reasoning as for every other distance-based method in this series:
# without this, features on larger scales would dominate the covariance
# matrix regardless of how informative they actually are.
X_scaled = StandardScaler().fit_transform(X)

# Create the model.
# n_components : how many principal components to keep — this is k from
#                the algorithm above. Can be an integer (e.g. 2), a float
#                between 0 and 1 (a target proportion of variance to
#                retain — scikit-learn then chooses k automatically), or
#                omitted entirely to keep all p components.
model = PCA(n_components=2)

# fit() computes the covariance matrix and its eigenvectors/eigenvalues
# (Steps 2-4) and selects the top k as the projection matrix W (Step 5).
model.fit(X_scaled)

# transform() applies Step 6 — the projection itself — producing the
# n x k matrix of scores.
X_pca = model.transform(X_scaled)

# fit_transform() does both fit() and transform() in a single call —
# the standard pattern when you only need to fit once.
X_pca = model.fit_transform(X_scaled)
```

---

#### Key parameters

| Parameter | Default | What it controls |
|---|---|---|
| `n_components` | `None` | How many principal components to keep ($k$). An integer keeps that many components; a float between 0 and 1 keeps as many components as needed to reach that proportion of total variance; `None` keeps all $p$ components (no dimensionality reduction, but still computes the rotation) |
| `svd_solver` | `'auto'` | The numerical method used to compute the eigen-decomposition — `'auto'` selects an appropriate method based on the data's shape; the choice doesn't change the result, only how it's computed |
| `random_state` | `None` | Seeds the randomness used by some `svd_solver` options, for reproducibility |

In practice, `n_components` is the only parameter that matters for most uses, and how to choose it is the subject of the next section.

---

#### Reading the output

```python
import numpy as np

# The projection matrix W from Step 5, with shape (k, p) — note this is
# the TRANSPOSE of W as written in the algorithm above (p, k). Each row
# is one principal component, written as a combination of the original
# features — these are the "loadings" referred to in Section 9's
# discussion of Use 2.
print(model.components_)

# The eigenvalues from Step 3-4, one per kept component, in descending
# order — how much variance each principal component captures, in the
# original units of the (standardised) data.
print(model.explained_variance_)

# The same eigenvalues, expressed as a PROPORTION of the total variance
# across ALL p original components (even though only k are kept) —
# this is the percentage figure used in Figure 11's annotation panel.
print(model.explained_variance_ratio_)

# The mean of each original feature, subtracted during standardisation —
# needed if you ever want to reverse the transformation.
print(model.mean_)

# X_pca itself — the n x k matrix of scores from Step 6.
print(X_pca.shape)
```

`explained_variance_ratio_` is usually the first thing worth checking: it tells you, directly, what fraction of the total variance in your dataset is captured by each principal component you kept, and by extension how much was *not* captured by reducing to $k$ dimensions.

---

#### Reversing the transformation

```python
# Project the scores back into the original p-dimensional space.
# This does NOT perfectly recover X_scaled unless k == p — some
# information was discarded when components beyond k were dropped.
X_reconstructed = model.inverse_transform(X_pca)
```

`inverse_transform()` is useful for visualising *how much* information was lost by keeping only $k$ components: comparing `X_reconstructed` to the original `X_scaled` shows exactly what the discarded components were capturing.

---

> ⚠️ **Warning:** the signs of `components_` (and therefore of `X_pca`'s columns) are arbitrary. An eigenvector $\mathbf{v}$ and its negation $-\mathbf{v}$ both satisfy $\mathbf{C}\mathbf{v} = \lambda\mathbf{v}$ equally well, so scikit-learn may return either. This means a principal component might point in the opposite direction to what you'd get from a different implementation, or from a previous run with slightly different data; the *direction* of variation is the same, but which way is "positive" along that axis can flip. This doesn't affect anything downstream (a classifier trained on flipped scores performs identically), but it can be confusing when comparing plots.

**PCA in Action**

The cell below loads the classic **Iris dataset**: 150 flowers, each described by 4 measurements (go back to Section 3 to read about the datasets used here). The `iris.data` is a $150 \times 4$ matrix, exactly the $\mathbf{X}$ from Section 9.1's algorithm: each row is one flower, and each column is one of the four original features below.

| Column | Feature | Meaning |
|---|---|---|
| 0 | `sepal length (cm)` | Length of the sepal (the leaf-like part beneath the petals) |
| 1 | `sepal width (cm)` | Width of the sepal |
| 2 | `petal length (cm)` | Length of the petal |
| 3 | `petal width (cm)` | Width of the petal |

After standardising, `PCA(n_components=2)` reduces this 4-feature dataset down to 2 principal components, following the `fit_transform()` pattern introduced above.

In [ ]:
# Listing 15.
# The Iris dataset: 150 flowers, each described by 4 measurements —
# sepal length, sepal width, petal length, and petal width (all in cm).
iris = load_iris()
X = iris.data
feature_names = iris.feature_names

# Standardise features first — Step 1 of the algorithm in Section 9.1.
X_scaled = StandardScaler().fit_transform(X)

# Reduce from 4 features down to 2 principal components.
model = PCA(n_components=2)
X_pca = model.fit_transform(X_scaled)

print('Original shape:', X.shape)
print('Reduced shape:', X_pca.shape)
print()

# How much of the total variance each of the 2 kept components captures,
# and how much they capture between them.
print('Explained variance ratio:', model.explained_variance_ratio_)
print('Total variance retained:', model.explained_variance_ratio_.sum())
print()

# Each row is one principal component, written as a combination of the
# 4 original features — the loadings discussed in Section 9.
print('Feature names:', feature_names)
print('Components (loadings):')
print(model.components_)

The printed output shows three things: the shape change from `(150, 4)` to `(150, 2)`, the dimensionality reduction itself; `explained_variance_ratio_`, showing that PC1 captures about 73% of the total variance and PC2 a further 23%, for 95.8% between them, meaning almost all of the information in the original 4 measurements is preserved in just 2 numbers per flower; and `components_`, the loadings for each of the 2 components across all 4 original features.

Look at the loadings for PC1: sepal length ($0.521$), petal length ($0.580$), and petal width ($0.565$) all have large, similarly-sized, positive loadings, while sepal width ($-0.269$) is smaller and has the opposite sign. This tells you that PC1 is essentially "overall flower size": a flower with large sepal length, petal length, and petal width (and slightly smaller sepal width) scores highly on PC1, and a smaller flower scores low. PC2 tells a different story: its loading on sepal width ($0.923$) dwarfs everything else (petal length and petal width are both close to $0.02$-$0.07$). PC2 is, almost entirely, a measure of sepal width on its own, independent of the "overall size" pattern that PC1 captured.

This is Use 2 from Section 9 in action: without being told anything about what these measurements represent, PCA has separated "how big is this flower overall" (PC1) from "how wide is its sepal, independent of its size" (PC2), two genuinely different patterns of variation, each captured by a single component.

The cell below visualises the effect of the transformation directly, colouring each of the 150 flowers by its species. The left panel plots two of the four original (standardised) features against each other: sepal length and sepal width. The right panel plots the same 150 flowers in the new PC1/PC2 space produced by `fit_transform()`.

Compare the two panels. In the original feature space on the left, the three species overlap substantially, *versicolor* and *virginica* in particular are tangled together, with sepal length and sepal width alone giving little indication that they're different species at all. In the PCA-reduced space on the right, the same three species separate out far more clearly: *setosa* forms its own distinct group along PC1, and while *versicolor* and *virginica* still overlap somewhat, the separation is noticeably better than in the original two features.

This is a direct illustration of Use 1 from Section 9: PC1 and PC2 together capture 95.8% of the variance across *all four* original measurements, not just the two shown on the left, so the right-hand plot is summarising more of the dataset's structure in two dimensions than the left-hand plot does, despite both being 2D scatter plots. The species labels were never used by PCA itself (it had no idea these groupings existed), yet the directions of greatest variance it found happen to align well with the differences between species, a common and useful property of PCA on real datasets.

In [ ]:
# Listing 16.
# Colour by species (iris.target) in both panels, so the same flowers
# can be tracked between the original feature space and the PCA-reduced
# space.
y = iris.target
species_names = iris.target_names

# Left panel: two of the original (standardised) features.
# Right panel: the data in the PCA-reduced space — PC1 vs PC2.
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
fig.canvas.header_visible = False
fig.canvas.toolbar_visible = False

for species in range(3):
    mask = y == species

    axes[0].scatter(X_scaled[mask, 0], X_scaled[mask, 1], s=30,
                     edgecolors='k', lw=0.3, alpha=0.7, label=species_names[species])
    axes[1].scatter(X_pca[mask, 0], X_pca[mask, 1], s=30,
                     edgecolors='k', lw=0.3, alpha=0.7, label=species_names[species])

axes[0].set_xlabel(feature_names[0])
axes[0].set_ylabel(feature_names[1])
axes[0].set_title('Original feature space\n(2 of 4 features shown)')
axes[0].legend(fontsize=8)
axes[0].grid(True, alpha=0.15)

axes[1].set_xlabel('PC1')
axes[1].set_ylabel('PC2')
axes[1].set_title('PCA-reduced space (2 of 4 dimensions)')
axes[1].legend(fontsize=8)
axes[1].grid(True, alpha=0.15)

plt.tight_layout()
plt.show()

### 10.1 Explained Variance and the Scree Plot

In the Iris example above, we chose `n_components=2` somewhat arbitrarily. It happened to retain 95.8% of the variance, but we picked $2$ before knowing that. In general, how do you decide how many components to keep?

The most common tool for this is the **scree plot**, a bar chart showing `explained_variance_ratio_` for every component, from PC1 down to the last one. Because `n_components=None` keeps *all* $p$ components (recall from Section 10's parameter table), fitting PCA this way lets you see the full picture: how much variance each and every component captures, before deciding how many to actually use.

Alongside the scree plot, it's common to plot the **cumulative explained variance**, the running total as you add each component one at a time (PC1 alone, then PC1+PC2, then PC1+PC2+PC3, and so on). This curve answers the practical question directly: "if I keep the first $k$ components, what percentage of the total variance have I retained?"

A widely used rule of thumb is to keep enough components to reach a chosen threshold on this cumulative curve, commonly 95%, though 90% or 99% are also used depending on how much information loss is acceptable for the task at hand. The scree plot and cumulative curve are usually shown together: the scree plot shows where the *individual* contributions drop off sharply (often called an "elbow", in the same spirit as the elbow method from Notebook 8), while the cumulative curve shows exactly where your chosen threshold is crossed.

In [ ]:
# Listing 17.
# Fit PCA with n_components=None, which keeps ALL p components — this is
# what lets us see the contribution of every component, not just the 2
# kept earlier.
model_full = PCA(n_components=None)
model_full.fit(X_scaled)

explained_ratio = model_full.explained_variance_ratio_
cumulative_var  = np.cumsum(explained_ratio)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.canvas.header_visible = False
fig.canvas.toolbar_visible = False

# ── Scree plot ───────────────────────────────────────────────────────────────
ax = axes[0]
ax.bar(range(1, len(explained_ratio)+1), explained_ratio*100,
       color='steelblue', edgecolor='white', linewidth=0.5)
ax.set_xlabel('Principal Component')
ax.set_ylabel('% Variance explained')
ax.set_title('Scree plot — look for the elbow')
ax.set_xticks(range(1, len(explained_ratio)+1))
ax.grid(True, alpha=0.3, axis='y')

for i, v in enumerate(explained_ratio):
    ax.text(i+1, v*100+0.5, f'{v*100:.1f}%', ha='center', fontsize=10)

# ── Cumulative variance ──────────────────────────────────────────────────────
ax = axes[1]
ax.plot(range(1, len(explained_ratio)+1), cumulative_var*100, 'o-',
        color='tomato', lw=2.5, ms=9)
ax.axhline(95, color='black', lw=1.5, linestyle='--', label='95% threshold')
ax.fill_between(range(1, len(explained_ratio)+1), 0, cumulative_var*100,
                alpha=0.15, color='tomato')
ax.set_xlabel('Number of PCs kept')
ax.set_ylabel('Cumulative % variance')
ax.set_title('Cumulative explained variance')
ax.set_xticks(range(1, len(explained_ratio)+1))
ax.set_ylim(0, 102)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle('Figure 12: Scree plot and cumulative variance — Iris dataset', fontsize=12, y=0.98)
plt.tight_layout()
plt.show()

# Report exactly how many PCs are needed to cross the 95% threshold.
print('Cumulative variance:')
for i, cv in enumerate(cumulative_var):
    marker = '<- 95% reached' if cv >= 0.95 and (i == 0 or cumulative_var[i-1] < 0.95) else ''
    print(f'  {i+1} PC(s): {cv*100:.2f}% {marker}')

### 10.2 Loadings: What Do the PCs Mean?

We saw `components_` already, in the very first PCA example: each row was a principal component, written as a list of numbers, one per original feature. Those numbers are called **loadings**, and Section 9 introduced what they mean: a loading near zero means that feature barely influences that PC, while a large positive or negative loading means that feature is a major contributor to the direction that PC points in.

For the Iris dataset, we already used these loadings to interpret what PC1 and PC2 represent: PC1's large, similarly-sized loadings on sepal length, petal length, and petal width, with sepal width pulling the opposite way, suggested PC1 captures something like "overall flower size". PC2's loading was dominated almost entirely by sepal width, suggesting it captures variation in sepal width that's largely independent of overall size. This is **Use 2** from Section 9: loadings let you attach a real-world meaning to an otherwise abstract new axis.

It's worth being precise about what loadings are and aren't used for. Loadings tell you how to *interpret* a PC, after it's been computed. They are **not** a tool for selecting which original features to keep. It might be tempting, having seen that PC2 is "mostly sepal width", to conclude you could get a similar result by just keeping the sepal width column and discarding the rest, but this isn't the same thing. A loading describes one feature's contribution to *one specific direction*, computed in the context of *all* the other features and their covariances; dropping features changes the covariance matrix entirely, and with it, every loading and every principal component.

If your goal is dimensionality reduction (Use 1), the only correct way to reduce your data is the projection from Section 9.1, $\mathbf{X}_{\text{PCA}} = \mathbf{X}_{\text{std}} \cdot \mathbf{W}$, using *all* of the original features, projected onto your chosen $k$ components. Loadings are a lens for understanding the result of that projection, not a shortcut for skipping it.

The cell below makes the loadings from `model.components_` explicit, arranging them in a table with one row per principal component and one column per original feature, then visualising both PC1 and PC2's loadings as bar charts, coloured blue for positive and red for negative, so the direction of each feature's contribution is visible at a glance. The final block walks through PC1's loadings one feature at a time, classifying each as a strong or weak, positive or negative contributor, the same reasoning used to interpret PC1 as "overall flower size" above.

In [ ]:
# Listing 18.
# Loadings — which original features drive each PC?
# model.components_ has shape (k, p): each row is one principal
# component, each column one original feature — exactly the layout
# needed for a DataFrame with PCs as rows and features as columns.
loadings_df = pd.DataFrame(
    model.components_,
    index=[f'PC{i+1}' for i in range(model.n_components_)],
    columns=feature_names,
)
print('Loadings:')
print(loadings_df.round(4).to_string())

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
fig.canvas.header_visible = False
fig.canvas.toolbar_visible = False

for i, ax in enumerate(axes):
    vals = loadings_df.iloc[i]
    # Colour bars by sign — blue for positive loadings, red for negative —
    # so the direction of each feature's contribution is visible at a glance.
    colours_l = ['steelblue' if v > 0 else 'tomato' for v in vals]
    bars = ax.bar(feature_names, vals, color=colours_l, edgecolor='white', lw=0.4)

    ax.axhline(0, color='black', lw=0.8)
    ax.set_title(f'PC{i+1} loadings')
    ax.set_ylabel('Loading coefficient')
    ax.tick_params(axis='x', rotation=20)
    ax.grid(True, alpha=0.25, axis='y')

    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, v + (0.01 if v > 0 else -0.03),
                f'{v:.2f}', ha='center', fontsize=9)

plt.suptitle('Figure 13: PCA loadings — original feature contributions (Iris)', fontsize=12, y=0.98)
plt.tight_layout()
plt.show()

# Interpret PC1's loadings: which features contribute strongly, and
# in which direction?
print('\nPC1 interpretation:')
for feat, load in zip(feature_names, loadings_df.iloc[0]):
    strength = 'strongly' if abs(load) > 0.4 else 'weakly'
    direction = 'POSITIVE' if load > 0 else 'NEGATIVE'
    print(f'  {feat}: {load:+.3f}  -> {strength} {direction}')



---


## 11. PCA with Breast Cancer Dataset

🔙 [Back to Table of Contents](#table-of-contents)

So far, every example has used the Iris dataset, small enough that 2 components already captured 95.8% of the variance, and small enough to sanity-check by eye. We now apply exactly the same workflow to a dataset where dimensionality reduction is doing real work: the **Breast Cancer Wisconsin (Diagnostic) Dataset**, described above, with 30 features per observation instead of 4.

> 📓 **Notebook recap:** scikit-learn's `PCA` computes its result using **Singular Value Decomposition (SVD)** internally, rather than directly finding the eigenvectors and eigenvalues of the covariance matrix as Section 8.1 did by hand. SVD is a more numerically stable way of arriving at the same answer; for our purposes, it's the same algorithm from Section 9.1, just computed via a more robust route, and `explained_variance_ratio_`, `components_`, and everything else behaves exactly as before.

Go back to Section 3 to remind yourself of what the breast cancer dataset contains.

### 11.1 Scree Plot and Variance Explained

With 30 original features, "how many components should I keep?" becomes a much more interesting question than it was for Iris's 4. The code below standardises the data, fits `PCA()` with `n_components=None` to see the contribution of all 30 components at once, and plots both the scree plot and the cumulative variance curve from Section 10.1, this time for 30 components rather than 4. It also calculates directly how many of those 30 components are needed to cross the 95% threshold, and reports the resulting **compression ratio**: the original number of features divided by the number of components kept. Before running it, it's worth asking yourself: with 30 features, would you expect to need most of them to capture 95% of the variance, or far fewer?

In [ ]:
# Listing 19.
# PCA with scikit-learn — breast cancer dataset (30 features)
# ──────────────────────────────────────────────────────────────────────────
# Load the dataset described above: 569 biopsies, 30 numeric features
# per biopsy, plus a binary target (0 = malignant, 1 = benign).
raw  = load_breast_cancer()
X_bc = raw.data    # shape (569, 30) — 569 samples x 30 features
y_bc = raw.target  # shape (569,)    — not used by PCA itself, kept for later

# Always standardise BEFORE PCA — Step 1 of the algorithm in Section 9.1.
# With 30 features on very different scales (e.g. "mean area" in the
# hundreds vs "mean smoothness" below 1), skipping this would let the
# largest-scaled features dominate the covariance matrix regardless of
# how informative they actually are.
scaler_bc = StandardScaler()
X_bc_s    = scaler_bc.fit_transform(X_bc)

# Fit PCA with n_components=None (the default), keeping ALL 30 components.
# This lets us see the full variance landscape across every component
# before deciding how many to keep — same approach as the scree plot
# for the Iris dataset above, but now for 30 features instead of 4.
pca_full = PCA()
pca_full.fit(X_bc_s)

explained  = pca_full.explained_variance_ratio_
cumulative = np.cumsum(explained)

# Find the smallest number of components whose cumulative variance
# reaches 95% — the rule of thumb from Section 10.1.
# np.searchsorted returns the 0-indexed position where 0.95 would be
# inserted to keep `cumulative` sorted; adding 1 converts this to "how
# many components", since cumulative[0] corresponds to 1 component.
n_95 = np.searchsorted(cumulative, 0.95) + 1

# ── Scree plot and cumulative variance, side by side ────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
fig.canvas.header_visible = False
fig.canvas.toolbar_visible = False

# Left panel: scree plot — % variance explained by each individual
# component, in descending order.
ax = axes[0]
ax.bar(range(1, len(explained)+1), explained*100,
       color='steelblue', edgecolor='none', alpha=0.85, width=0.8)
ax.axvline(n_95, color='red', lw=2, linestyle='--', label=f'{n_95} PCs for 95%')
ax.set_xlabel('Principal Component')
ax.set_ylabel('% Variance explained')
ax.set_title('Scree plot — breast cancer dataset (30 features)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.25, axis='y')

# Right panel: cumulative variance — the running total as components are
# added one at a time, same as the Iris cumulative plot above.
ax = axes[1]
ax.plot(range(1, len(cumulative)+1), cumulative*100, 'o-', color='tomato', lw=2, ms=5)
ax.axhline(95, color='black', lw=1.5, linestyle='--', label='95% threshold')
ax.axvline(n_95, color='red', lw=2, linestyle='--', label=f'{n_95} PCs needed')
ax.set_xlabel('Number of PCs')
ax.set_ylabel('Cumulative % variance')
ax.set_title('Cumulative explained variance')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.25)
ax.set_ylim(0, 102)

plt.suptitle('Figure 14: PCA on breast cancer dataset — how many components?', fontsize=12, y=0.98)
plt.tight_layout()
plt.show()

# Summarise the dimensionality reduction achieved by keeping only the
# components needed to reach 95% of the total variance.
print(f'Original features:              {X_bc.shape[1]}')
print(f'Components for 95% variance:    {n_95}')
print(f'Compression ratio:              {X_bc.shape[1]/n_95:.1f}x')

---

The result is striking: although the dataset has 30 original features, only **10 principal components** are needed to capture 95% of the total variance, a **3.0x compression ratio**. Look at the scree plot to see why: the first few components account for the vast majority of the variance (PC1 alone captures around 44%), and the bars drop off quickly, with the components beyond around 10 contributing almost nothing individually.

This makes sense in light of how the dataset was constructed. Recall from the feature table earlier that the 30 features come in three groups of 10, `mean`, `se` (standard error), and `worst`, all computed from the *same* underlying measurements of the *same* cell nuclei (radius, texture, perimeter, area, and so on). A tumour with large mean radius will tend to also have a large "worst radius" and a correspondingly large mean area, mean perimeter, and so on; these 30 features are heavily correlated with each other, in the same way feature_2 and feature_3 were correlated with feature_1 in the very first PCA example. PCA has effectively "noticed" this redundancy and folded it into a much smaller set of components.

This is dimensionality reduction delivering a genuinely practical result: a classifier trained on these 10 components, rather than the original 30 features, would be working with a third of the original input size, while still having access to 95% of the information those 30 features contained. We'll put this into practice next.

### 11.2 The Biplot: Data in PC Space

The 10 components needed for 95% of the variance are still too many to plot directly, but $k=2$ gives us something we can actually look at: a scatter plot where each of the 569 biopsies is reduced to just two numbers, its position along PC1 and PC2. This combination of a 2D PCA scatter with the original features' loadings drawn on top is called a **biplot**.

The left panel below is the scatter alone, coloured by diagnosis (the `target` column from Section 11's feature table). PCA was never told which biopsies were malignant and which were benign, so if the two classes separate visibly along PC1 and PC2, that's a strong sign that the directions of greatest variance in this dataset happen to align with the difference between malignant and benign tumours.

The right panel overlays **loading arrows**: each arrow represents one original feature, pointing in the direction of $(\text{PC1 loading}, \text{PC2 loading})$ for that feature, the same loadings from Section 10.2, just for two specific features at a time instead of all 30. A long arrow means that feature has a large loading on PC1 and/or PC2, i.e. it's a major contributor to one or both of these components; the *direction* an arrow points in tells you which combination of PC1 and PC2 that feature aligns with.

With 30 features, plotting all 30 arrows would be unreadable: many of them point in nearly the same direction, since (as discussed in Section 11.1) many of these features measure closely related things like size and shape. The code below selects a handful of features whose loading arrows point in genuinely *different* directions, so that each arrow and its label can be told apart, rather than simply picking the 5 or 6 features with the single largest loading on PC1 (which, as you might guess from Section 10.2's PC1 interpretation, would mostly point the same way).

In [ ]:
# Listing 20.
# The biplot: 2-D PC space with loading arrows
# ──────────────────────────────────────────────────────────────────────────
# Fit a fresh PCA with only 2 components — separate from pca_full above,
# since we now want the actual 2D scores (X_bc_2d), not just the
# variance ratios.
pca_2   = PCA(n_components=2)
X_bc_2d = pca_2.fit_transform(X_bc_s)

pc1_var = pca_2.explained_variance_ratio_[0] * 100
pc2_var = pca_2.explained_variance_ratio_[1] * 100

fig, axes = plt.subplots(1, 2, figsize=(10, 6))
fig.canvas.header_visible = False
fig.canvas.toolbar_visible = False

# ── Left panel: the 2D PCA scatter, coloured by diagnosis ───────────────────
ax = axes[0]
for cls, col, lbl in [(0, 'tomato', 'Malignant'), (1, 'steelblue', 'Benign')]:
    m = y_bc == cls
    ax.scatter(X_bc_2d[m, 0], X_bc_2d[m, 1], color=col, s=25, alpha=0.6,
               edgecolors='k', lw=0.15, label=lbl)

ax.set_xlabel(f'PC1 ({pc1_var:.1f}%)')
ax.set_ylabel(f'PC2 ({pc2_var:.1f}%)')
ax.set_title('PCA scatter\nClasses separate well in 2D PC space')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.2)

# ── Right panel: the same scatter, faded, with loading arrows overlaid ──────
ax = axes[1]
for cls, col in [(0, 'tomato'), (1, 'steelblue')]:
    m = y_bc == cls
    # Heavily faded (alpha=0.25) and smaller — the data points are now just
    # context for the arrows, not the focus of this panel.
    ax.scatter(X_bc_2d[m, 0], X_bc_2d[m, 1], color=col, s=12, alpha=0.25, edgecolors='none')

# loadings_bc has shape (30, 2): one row per original feature, columns are
# its loading on PC1 and PC2 respectively — i.e. components_.T.
loadings_bc = pca_2.components_.T

# Selecting the 6 features with the single largest PC1 loading tends to
# pick features that are themselves highly correlated with each other
# (e.g. radius, perimeter, and area all measure "size"), so their arrows
# all point in almost the same direction and their labels overlap.
#
# Instead, select features whose loading VECTORS point in genuinely
# different directions: sort by overall loading magnitude
# (sqrt(PC1_loading^2 + PC2_loading^2)), then greedily keep a feature only
# if its direction is at least MIN_ANGLE_DEG away from every feature
# already selected. This favours a small set of arrows that are visually
# distinguishable, at the cost of not necessarily being the single largest
# loadings on PC1 alone.
mag    = np.linalg.norm(loadings_bc, axis=1)
angles = np.arctan2(loadings_bc[:, 1], loadings_bc[:, 0])
order  = np.argsort(mag)[::-1]

MIN_ANGLE_DEG = 20
selected = []
for idx in order:
    angle_deg = np.degrees(angles[idx])
    # Angular distance to each already-selected direction, wrapped to
    # the range [0, 180] so that e.g. 170 deg and -170 deg count as 20
    # deg apart, not 340 deg apart.
    far_enough = all(
        abs(((angle_deg - np.degrees(angles[s]) + 180) % 360) - 180) >= MIN_ANGLE_DEG
        for s in selected
    )
    if far_enough:
        selected.append(idx)
    if len(selected) == 5:
        break

# scale stretches the (unit-length-ish) loading vectors so the arrows are
# visible against the PC score axes, which are on a much larger scale.
scale = 9
for idx in selected:
    x, y = loadings_bc[idx] * scale
    ax.annotate('', xy=(x, y), xytext=(0, 0),
                arrowprops=dict(arrowstyle='->', color='goldenrod', lw=1.8))
    # Label placed slightly beyond the arrowhead, along the same direction —
    # since arrows now point in distinct directions, labels no longer
    # collide with each other.
    ax.annotate(raw.feature_names[idx], xy=(x, y), xytext=(x*1.3, y*1.3),
                fontsize=8, color='#7a5c00', ha='center', va='center')

ax.set_xlabel(f'PC1 ({pc1_var:.1f}%)')
ax.set_ylabel(f'PC2 ({pc2_var:.1f}%)')
ax.set_title('Biplot — 5 most distinct loading directions')
ax.grid(True, alpha=0.2)

plt.suptitle('Figure 15: PCA biplot — breast cancer dataset', fontsize=12, y=0.98)
plt.tight_layout()
plt.show()

### 11.3 Classifying in PCA Space — The Right Approach

The biplot above showed that malignant and benign tumours separate well in PC space, which suggests a classifier trained on PC1 and PC2 (or however many components you keep) could work well too. But it's important to be precise about what "training a classifier on PC space" actually involves, because PCA introduces a step that's easy to get wrong.

Once PCA has been fitted, it isn't just a one-off calculation you did to make a nice plot, it's now part of your **pipeline**, in exactly the same sense as the `StandardScaler` from earlier. Every single example your classifier ever sees, during training, during testing, and after deployment, has to pass through both of these steps, *in the same order, using the same fitted objects*, before the classifier sees it. A new sample arriving with 30 raw measurements cannot be passed to the classifier directly; it has to go through:

1. **Standardise**, using the *same* `StandardScaler` instance that was `fit()` on the training data, not a new one fitted on this sample, and not one fitted on a combination of training and test data.
2. **Project**, using the *same* `PCA` instance that was `fit()` on the (standardised) training data, again, not refitted.
3. **Classify**, passing the resulting $k$-dimensional scores, not the original 30 features, to the classifier.

The critical phrase in both of the first two steps is "the *same* ... that was fit on the training data". `StandardScaler.fit()` computes a mean and standard deviation from whatever data you give it; `PCA.fit()` computes a covariance matrix, eigenvectors, and eigenvalues from whatever data you give it. If you fit either of these using your test data, even just to "standardise everything together for convenience", information about the test set has leaked into the numbers used to transform the training data. This is called **data leakage**, and the practical consequence is that your classifier's reported accuracy on the test set will be *higher* than it would be on genuinely new, unseen data, because the test set wasn't really unseen by the preprocessing steps at all.

The fix is simple in principle, if easy to get wrong in practice: call `fit()` (or `fit_transform()`) only once, on the training data, for both the `StandardScaler` and the `PCA`. For every other dataset, the test set, a validation set, or a single new sample arriving later, call only `transform()`, never `fit()` again. The next section puts this into practice.

In [ ]:
# Listing 21.
# Classifying in PCA space — correct train/test pipeline
# ──────────────────────────────────────────────────────────────────────────
# Split FIRST, then fit all preprocessing on TRAIN only — this is the
# rule from Section 11.3, applied to both the scaler and PCA below.
X_tr, X_te, y_tr, y_te = train_test_split(X_bc, y_bc, test_size=0.2, random_state=0)

# Standardise: fit on TRAIN, then transform both TRAIN and TEST using
# that same fitted scaler. X_te never influences scaler_clf's mean or
# standard deviation.
scaler_clf = StandardScaler()
X_tr_s = scaler_clf.fit_transform(X_tr)   # fit + transform on TRAIN
X_te_s = scaler_clf.transform(X_te)       # transform only (no refit)

# For each candidate number of components, fit a fresh PCA on the
# standardised TRAINING data only, project both TRAIN and TEST through
# it, then train and evaluate a classifier in that reduced space.
results = {}
for n_comp in [2, 5, 10, 20, 30]:
    pca_clf  = PCA(n_components=n_comp)
    X_tr_pca = pca_clf.fit_transform(X_tr_s)  # fit + project TRAIN
    X_te_pca = pca_clf.transform(X_te_s)      # project TEST (no refit)

    # max_iter=1000 gives the solver enough iterations to converge on
    # this dataset — the default (100) can fall short on standardised
    # data with many features.
    clf = LogisticRegression(max_iter=1000, random_state=0)
    clf.fit(X_tr_pca, y_tr)

    results[n_comp] = accuracy_score(y_te, clf.predict(X_te_pca))

# Baseline: a classifier trained directly on all 30 standardised
# features, with no PCA step at all — n_comp=30 above should match this
# closely, since keeping all components is (up to rotation) equivalent
# to not reducing dimensionality at all.
clf_full = LogisticRegression(max_iter=1000, random_state=0)
clf_full.fit(X_tr_s, y_tr)
acc_full = accuracy_score(y_te, clf_full.predict(X_te_s))

# ── Plot accuracy vs number of components kept ──────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
fig.canvas.header_visible = False
fig.canvas.toolbar_visible = False

ax.plot(list(results.keys()), list(results.values()),
        'o-', color='steelblue', lw=2, ms=8, label='PCA + Logistic Regression')
ax.axhline(acc_full, color='tomato', lw=2, linestyle='--',
           label=f'All 30 features (no PCA): {acc_full:.1%}')

ax.set_xlabel('Number of PCs')
ax.set_ylabel('Test accuracy')
ax.set_title('Classification accuracy vs number of PCs')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

plt.suptitle('Figure 16: Accuracy vs dimensionality — breast cancer classification', fontsize=12, y=0.98)
plt.tight_layout()
plt.show()

# Print every result, including the no-PCA baseline, for direct comparison.
for n, acc in results.items():
    print(f'  {n:2d} PCs: {acc:.1%}')
print(f'  30 features: {acc_full:.1%}')

---

These results tell a reassuring story. With just **2 components** (the same 2 used for the biplot), a logistic regression classifier already reaches 92.1% accuracy, consistent with how cleanly the two classes separated visually in Figure 15. Accuracy then climbs to **95.6% with 5 components**, dips slightly to 94.7% at 10 (a reminder that more components isn't *always* strictly better for a given classifier; some of what's added between 5 and 10 components may be more noise than signal for this particular model), and reaches **96.5% from 20 components onward**.

The last two rows are the important comparison: **30 PCs (96.5%)** and **30 original features, no PCA at all (96.5%)** give *identical* accuracy. This is exactly what Section 11.3's framing predicts: with $k=p=30$, PCA hasn't discarded anything, it's only rotated the data into a new coordinate system, and a linear classifier like logistic regression is unaffected by such a rotation. PCA only starts to matter once $k < p$, at which point it's discarding the components with the smallest eigenvalues, directions where, by definition, the data barely varies.

Recall from Section 11.1 that 10 components were needed to capture 95% of the *variance*. Here, just 5 components already reach 95.6% *accuracy*, a useful reminder that "variance captured" and "usefulness for a specific classification task" are related but not identical measures. A direction with very little variance could, in principle, still happen to align closely with the boundary between classes; conversely, a high-variance direction might capture variation that has nothing to do with the distinction the classifier cares about. In this case, though, the two measures broadly agree: a handful of components captures most of both the variance and the classification accuracy, while the remaining ~25 components contribute very little to either.


### 11.4 Common PCA Mistakes

The most common PCA mistake is **forgetting to standardise**, a mistake easy to make by accident, since `PCA().fit(X)` runs without complaint either way; nothing in scikit-learn forces you to standardise first. The code below demonstrates exactly what goes wrong when you skip this step, by fitting PCA on the *raw* `X_bc` (no `StandardScaler`) alongside the standardised `X_bc_s` used everywhere else in this section, and comparing the results side by side.

Look back at the feature table for the breast cancer data in Section 3: `mean area` ranges from roughly 140 to 2500, and `worst area` similarly into the thousands, while features like `mean smoothness` or `mean fractal dimension` range from roughly 0.05 to 0.16, a difference of several orders of magnitude.

Recall from Section 6.1 that variance is calculated in the *squared* units of the original feature, so a feature measured in the thousands contributes a variance on the order of *millions*, while a feature measured in tenths contributes a variance on the order of hundredths. When PCA computes the covariance matrix from unscaled data, these enormous differences in scale completely dominate it: PC1 ends up pointing almost entirely in the direction of whichever features happen to have the largest raw numbers, regardless of whether those features are actually the most informative ones for distinguishing malignant from benign tumours.

The printed output makes this concrete: without standardisation, PC1 alone captures over 98% of the "variance", but this isn't 98% of the dataset's *information*, it's 98% of the dataset's *millimetre-and-area-scale numbers*. With standardisation, every feature is rescaled to have the same variance (1.0) before PCA ever sees it, so PC1's 44.3% reflects genuine shared structure across features, not an artefact of which original units happened to produce the largest numbers. The takeaway is the one stated back in Section 9.1, Step 1: standardisation isn't an optional cleanup step you can skip for convenience, without it, PCA's output is dominated by units of measurement rather than by the structure of your data.

In [ ]:
# Listing 22.
# PCA pitfall: what happens when you forget to standardise?
# ──────────────────────────────────────────────────────────────────────────
# Two PCA models, fitted on the SAME underlying data — one on the raw,
# unscaled X_bc, and one on the standardised X_bc_s used everywhere else
# in this section.
pca_no_scale = PCA(n_components=2)
pca_scaled   = PCA(n_components=2)

X_bc_2d_noscale = pca_no_scale.fit_transform(X_bc)    # raw data — WRONG
X_bc_2d_scaled  = pca_scaled.fit_transform(X_bc_s)    # standardised — correct

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
fig.canvas.header_visible = False
fig.canvas.toolbar_visible = False

for ax, X_plot, pca_obj, title in [
    (axes[0], X_bc_2d_noscale, pca_no_scale, 'WITHOUT standardisation\n(large-scale features dominate)'),
    (axes[1], X_bc_2d_scaled,  pca_scaled,   'WITH standardisation\n(all features contribute equally)'),
]:
    for cls, col, lbl in [(0, 'tomato', 'Malignant'), (1, 'steelblue', 'Benign')]:
        m = y_bc == cls
        ax.scatter(X_plot[m, 0], X_plot[m, 1], color=col, s=20, alpha=0.6,
                   edgecolors='k', lw=0.1, label=lbl)

    v1 = pca_obj.explained_variance_ratio_[0] * 100
    v2 = pca_obj.explained_variance_ratio_[1] * 100

    ax.set_xlabel(f'PC1 ({v1:.1f}%)')
    ax.set_ylabel(f'PC2 ({v2:.1f}%)')
    ax.set_title(title)
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.2)

plt.suptitle('Figure 17: Always standardise before applying PCA!', fontsize=12, y=0.98)
plt.tight_layout()
plt.show()

# Compare PC1's share of variance with and without standardisation.
print('Without standardisation: PC1 =', round(pca_no_scale.explained_variance_ratio_[0]*100, 1), '%')
print('With standardisation:    PC1 =', round(pca_scaled.explained_variance_ratio_[0]*100, 1), '%')

# Without standardisation, PC1's loadings are dominated by whichever
# feature happens to have the largest raw numerical range — show which
# feature that is, and how large its loading is compared to the rest.
idx = np.argmax(np.abs(pca_no_scale.components_[0]))
print(f'\nLargest PC1 loading (no standardisation): '
      f'{raw.feature_names[idx]} = {pca_no_scale.components_[0][idx]:.3f}')


---

**PCA checklist — carry this into every project:**

- Always **standardise** (zero mean, unit variance) before applying PCA. Section 11.4 showed what happens without this: PC1 can end up dominated entirely by whichever feature has the largest raw numerical range, regardless of how informative it actually is.
- **Fit on training data only**, both the `StandardScaler` and the `PCA` itself. Fitting either on test data is data leakage (Section 11.3), and produces test accuracy figures that won't hold up on genuinely new data.
- Use the **scree plot and cumulative variance curve** (Section 11.1) to choose $k$, but remember that "variance captured" and "useful for your specific task" aren't always the same thing (Section 11.3): always validate your choice of $k$ against your actual downstream task, not just the cumulative variance percentage.
- Project new data using the **already-fitted** PCA: call `transform()`, never `fit()` or `fit_transform()`, on anything other than the original training set.
- Use **loadings** (Section 10.2) to interpret what each PC means in terms of the original features, but remember loadings describe a feature's contribution to *one specific direction in the context of all the others*; they are not a substitute for the projection itself, and dropping low-loading features directly is not the same as PCA's dimensionality reduction.
- Remember: principal components are **linear combinations of the original features**, a PC is "a bit of this feature, plus a bit of that one, minus a bit of another", not a single measurable quantity. Sometimes this combination has a clear real-world interpretation (Section 10.2's "overall flower size"), but often it won't, so don't force a physical meaning onto a PC where none exists.


---


## 12. Summary

🔙 [Back to Table of Contents](#table-of-contents)

### OPTICS

| Concept | Description |
|---------|-------------|
| Reachability distance | $\max(\text{core-dist}(o),\ d(p, o))$ — smooths intra-cluster distances |
| Core distance | Distance from a point to its `minPts`-th nearest neighbour |
| Reachability plot | Orders all points by reachability; valleys = clusters, peaks = noise |
| Advantage over DBSCAN | Handles clusters of varying density without a single fixed $\varepsilon$ |

### Building Blocks of PCA

| Concept | What it is | Why it matters for PCA |
|---------|-----------|------------------------|
| Variance | Spread of a single variable | Tells us how informative a feature is |
| Covariance | Co-movement of two variables | Reveals redundancy between features |
| Covariance matrix | All pairwise covariances ($p \times p$) | Starting point for PCA |
| Eigenvector | Direction unchanged by matrix multiplication | Principal component axis |
| Eigenvalue | Scaling factor for the eigenvector | Variance along that PC direction |

### PCA

| Concept | Description |
|---------|-------------|
| Algorithm | Standardise → covariance matrix → eigenvectors → sort → select top $k$ → project |
| Score | Projected coordinate of a data point in PC space |
| Loading | Coefficient of an original feature within a PC eigenvector |
| Scree plot | Bar chart of % variance per PC; look for an elbow |
| Cumulative variance | Running total; keep enough PCs to reach 95% (rule of thumb) |
| Data leakage | Fitting scaler or PCA on test data — always fit on training data only |

**The single most important practical rule:**  
Fit the scaler and PCA on **training data only**. Transform test data using the fitted objects. Never the other way round.



---


## 13. References

🔙 [Back to Table of Contents](#table-of-contents)

Ankerst, M., Breunig, M. M., Kriegel, H.-P. and Sander, J. (1999) ‘OPTICS: ordering points to identify the clustering structure’, *ACM SIGMOD Record*, 28(2), pp. 49–60.

Ester, M., Kriegel, H.-P., Sander, J. and Xu, X. (1996) ‘A density-based algorithm for discovering clusters in large spatial databases with noise’, in *Proceedings of the Second International Conference on Knowledge Discovery and Data Mining (KDD-96)*, Portland, OR, pp. 226–231.

Jolliffe, I. T. (2002) *Principal Component Analysis*. 2nd edn. New York: Springer.

Pedregosa, F. *et al.* (2011) ‘Scikit-learn: machine learning in Python’, *Journal of Machine Learning Research*, 12, pp. 2825–2830. Available at: https://scikit-learn.org (Accessed: May 2026).

Pearson, K. (1901) ‘On lines and planes of closest fit to systems of points in space’, *Philosophical Magazine*, 2(11), pp. 559–572.

Strang, G. (2016) *Introduction to Linear Algebra*. 5th edn. Wellesley, MA: Wellesley-Cambridge Press.